# Copyright (c) 2026 hidemi-k / Licensed under the MIT License

# NETCONF × Agentic RAG Phase 7 - Orchestrator-Worker パターン

## 🎯 Phase 6の目標

```
Phase 5: [inventory] → Generator(ReAct) → validate → fix → deploy → audit  ← 単体操作のみ
Phase 6: Orchestrator が複数操作を DAG に分解 → Worker(Phase5) へ順次ディスパッチ
```

### 変化のポイント：単体操作 → 複数操作
| | Phase 5 | Phase 6 |
|---|---|---|
| 対応操作数 | 1 操作のみ | 複数操作（混在）対応 |
| タスク管理 | なし | DAG による依存関係解決 |
| 制御レイヤー | Worker のみ | Orchestrator + Worker |
| 失敗時 | Worker が rollback | Orchestrator が残タスクを中断 |

### v2 修正（実機投入対応）
- `step1_translate_query`: ASCII比率で英語を自動検出 → 翻訳をバイパス
  （Orchestratorから英語クエリを渡した場合に翻訳が崩壊する問題を解決）
- `result_aggregator`: 全スキップ時に `dry_run_complete` を返すよう改善

### 新規追加 Skills（Phase 6）
- **`task_decomposer`**: 自然言語入力を操作リスト(JSON)に分解する LLM ベース Skill
- **`dependency_resolver`**: タスクリストをトポロジカルソートし実行順序を決定する Skill
- **`result_aggregator`**: 全 Task の結果を集約してユーザー向け最終レポートを生成する Skill

### Reviewer の修正（Phase 5 バグ修正）
- Junos 仕様判断をプロンプトから削除 → validate_xml Skill に完全移譲
- Reviewer は「ユーザー意図との整合性チェック」のみに限定


## 📦 ライブラリのインポート

In [1]:
import asyncio
import os
import re
import configparser
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
from typing import Optional, List, Dict, Any, Callable
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
# ★ v2: openai.OpenAI は不要（OpenAIChatClient が内部で処理）

# LangChain/FAISS
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
# ★ v2: langchain_groq.ChatGroq → langchain_openai.ChatOpenAI (Groq互換)
from langchain_openai import ChatOpenAI

# Junos
try:
    from jnpr.junos import Device
    from jnpr.junos.utils.config import Config
    from jnpr.junos.exception import RpcError, ConnectError, ConfigLoadError, CommitError
    JUNOS_AVAILABLE = True
    print("✅ Junos modules available")
except ImportError:
    JUNOS_AVAILABLE = False
    print("⚠️ Junos modules not found")

# agent-framework rc3
# ★ v2: BaseChatClient / ChatResponse は不要（手動実装を廃止）
from agent_framework import (
    Agent,
    Message
)
from agent_framework.openai import OpenAIChatClient  # ★ MAFネイティブ

print("✅ インポート完了")
print("   GroqChatClient (手動実装) → OpenAIChatClient (MAFネイティブ) に移行")


✅ Junos modules available
✅ インポート完了
   GroqChatClient (手動実装) → OpenAIChatClient (MAFネイティブ) に移行


## 🔑 APIキー読み込み

In [2]:
config = configparser.ConfigParser()
config_paths = [
    '/home/admin/config/config.ini',
    './config.ini',
    os.path.expanduser('~/config/config.ini')
]

GROQ_API_KEY = None
for path in config_paths:
    if os.path.exists(path):
        config.read(path)
        if 'GROQ' in config and 'GROQ_API_KEY' in config['GROQ']:
            GROQ_API_KEY = config['GROQ']['GROQ_API_KEY']
            print(f"✅ APIキーを {path} から読み込みました")
            break

if not GROQ_API_KEY:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("✅ APIキーを環境変数から読み込みました")

# ★ v2: os.environ["GROQ_API_KEY"] のセットを削除
#        ChatGroq を廃止し ChatOpenAI(api_key=...) で直接渡すため不要
print("✅ APIキー設定完了")


✅ APIキーを /home/admin/config/config.ini から読み込みました
✅ APIキー設定完了


## 🔍 RAGコンポーネント

In [3]:
print("📚 ベクトルストアを読み込み中...")

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
FAISS_DB_PATH = "/home/admin/faiss_db"

if os.path.exists(FAISS_DB_PATH):
    vectorstore = FAISS.load_local(
        FAISS_DB_PATH,
        embedding,
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    print(f"✅ ベクトルストア読み込み完了")
else:
    print(f"⚠️ ベクトルストアが見つかりません")
    retriever = None

# ★ v2: ChatGroq → ChatOpenAI (Groq互換エンドポイント)
#        langchain_groq が不要になり、langchain_openai に統一
GROQ_BASE_URL  = "https://api.groq.com/openai/v1"
DEFAULT_MODEL  = "llama-3.3-70b-versatile"

llm = ChatOpenAI(
    model=DEFAULT_MODEL,
    temperature=0,
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL,
)
print("✅ RAG準備完了")
print(f"   llm: ChatOpenAI (Groq互換) / {DEFAULT_MODEL}")


📚 ベクトルストアを読み込み中...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ベクトルストア読み込み完了
✅ RAG準備完了
   llm: ChatOpenAI (Groq互換) / llama-3.3-70b-versatile


## 🤖 OpenAIChatClient ファクトリ（MAFネイティブ）

### v2 変更点（Phase 5 継承）
- `GroqChatClient(BaseChatClient)` 約40行の手動実装を廃止
- `make_client()` 4行のファクトリ関数に置き換え
- MAFが内部で処理: Native型変換 / ChatResponse組み立て / エラーハンドリング

### Phase 6 での利用
- Orchestrator Agent: `make_client()` で生成
- 各 Worker 内 Generator/Reviewer: 既存の `make_client()` をそのまま使用
- モデル切り替えも `make_client(model_id=...)` 1行で完結

### モデル指定
- `llama-3.3-70b-versatile`（推奨・デフォルト）
- `llama-3.1-8b-instant`（高速）


In [4]:
# ★ v2: GroqChatClient クラス（約40行の手動実装）を廃止
#        MAFネイティブ OpenAIChatClient のファクトリ関数に置き換え
#
# 削除された処理（MAF内部で自動処理される）:
#   - role の .value 取得など Native型変換
#   - msg.text / msg.contents のフォールバック処理
#   - ChatResponse.from_dict(...) の手動組み立て
#   - agent_name による会話履歴への手動追記

def make_client(model_id: str = DEFAULT_MODEL) -> OpenAIChatClient:
    """
    MAFネイティブ OpenAIChatClient を生成する。

    Groq は OpenAI 互換 API のため base_url を差し替えるだけで流用可能。
    BaseChatClient の手動実装（_inner_get_response・型変換）は一切不要。

    引数名は MAF 公式仕様準拠:
      model_id : モデル名（OpenAI SDK の 'model' ではなく MAF 統一仕様）
      api_key  : APIキー
      base_url : Groq 互換エンドポイント
    """
    return OpenAIChatClient(
        model_id=model_id,
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
    )

print("✅ make_client() 定義完了 (GroqChatClient 廃止)")
print(f"   エンドポイント  : {GROQ_BASE_URL}")
print(f"   デフォルトモデル: {DEFAULT_MODEL}")


✅ make_client() 定義完了 (GroqChatClient 廃止)
   エンドポイント  : https://api.groq.com/openai/v1
   デフォルトモデル: llama-3.3-70b-versatile


## 🛠️ Phase 6: Skill定義

### Skill一覧（累計 10 個）

| Skill | 役割 | Phase | 変更 |
|---|---|---|---|
| `validate_xml` | XML構造検証 | 3継承 | |
| `fix_xml` | 自動修正 | 3継承 | |
| `deploy_netconf` | NETCONFデプロイ | 3継承 | |
| `rollback` | candidate破棄/rescue復元 | 4継承 | |
| `audit` | デプロイ後の反映確認 | 4継承 | |
| `get_inventory` | デバイスのVLAN一覧取得 | 5継承 | |
| `lookup_documentation` | RAG検索Skill化 | 5継承 | |
| `task_decomposer` | タスク分割・DAG生成 | **6 NEW** | ★ |
| `dependency_resolver` | 依存関係解決・実行順序決定 | **6 NEW** | ★ |
| `result_aggregator` | 全Taskの結果集約・最終報告 | **6 NEW** | ★ |


In [5]:
# ============================================================
# 🛠️ Skill定義
# Phase 2のメソッドを独立したSkill関数として再定義
# ============================================================

@dataclass
class Skill:
    """
    エージェントが使用できる能力（Skill）の定義
    
    Attributes:
        name: Skill名（エージェントの instructions 内で参照される）
        description: エージェントへの説明（いつ・何に使うか）
        function: 実際に呼び出される関数
        parameters: パラメータ定義（型・説明）
    """
    name: str
    description: str
    function: Callable
    parameters: Dict[str, Any] = field(default_factory=dict)

    def execute(self, **kwargs) -> Any:
        """Skillを実行する"""
        return self.function(**kwargs)

    def __str__(self):
        return f"Skill(name={self.name}, description={self.description[:50]}...)"


# ============================================================
# Skill 1: XML構造検証
# Phase 2の _validate_xml() を強化・独立化
# ============================================================
def validate_xml_structure(xml_config: str) -> Dict[str, Any]:
    """
    Junos XML構造を詳細検証する（v5強化版）

    変更点:
    - <vlan-id> を削除操作内では warning → error に昇格
      （Junosはdelete操作に<vlan-id>があると拒否するため）
    - <vlans>欠落チェックのエラーメッセージを明確化
    - <name>/<n> 両方を許容（v4継承）
    """
    if not xml_config or not xml_config.strip():
        return {"valid": False, "errors": ["XML is empty"], "warnings": []}

    try:
        root = ET.fromstring(xml_config)
        errors = []
        warnings = []

        # ① ルート要素チェック
        if root.tag != 'configuration':
            errors.append(f"Root must be <configuration>, got <{root.tag}>")

        # ② 数字のみのタグ名を禁止
        for elem in root.iter():
            if elem.tag.isdigit():
                errors.append(f"Numeric-only tag name forbidden: <{elem.tag}>")

        # ③ <vlan> が直接 <configuration> 下にある → <vlans> 欠落 [ERROR]
        if root.find('vlan') is not None:
            errors.append("<vlan> must be under <vlans>: missing <vlans> parent tag")

        # ④ <operation>タグ形式（子タグ）を検出 → エラー [ERROR]
        for vlan in root.findall('.//vlan'):
            op_tag = vlan.find('operation')
            if op_tag is not None:
                errors.append(
                    "<operation> must be an XML attribute, not a child tag. "
                    'Use: <vlan operation="delete"> instead of <operation>delete</operation>'
                )

        # ⑤ 削除操作の構造チェック
        for vlan in root.findall('.//vlan[@operation="delete"]'):
            # [ERROR] <vlan-id> は削除操作で Junos が拒否する → error に昇格
            if vlan.find('vlan-id') is not None:
                errors.append(
                    "Delete operation must NOT contain <vlan-id>: "
                    "Junos rejects delete with vlan-id present"
                )
            # [ERROR] VLAN名タグが必須 (<name> または <n> を許容)
            has_name = vlan.find('name') is not None or vlan.find('n') is not None
            if not has_name:
                errors.append("Delete operation must have <name> (or <n>) tag with VLAN name")

        # ⑥ 作成操作の構造チェック（情報のみ → warnings）
        for vlan in root.findall('.//vlan'):
            if vlan.get('operation') != 'delete' and vlan.find('operation') is None:
                has_name = vlan.find('name') is not None or vlan.find('n') is not None
                if not has_name:
                    warnings.append("Create operation: missing <name> tag")
                if vlan.find('vlan-id') is None:
                    warnings.append("Create operation: missing <vlan-id> tag")

        return {
            "valid": len(errors) == 0,
            "errors": errors,
            "warnings": warnings
        }

    except ET.ParseError as e:
        return {
            "valid": False,
            "errors": [f"XML syntax error: {str(e)}"],
            "warnings": []
        }

# ============================================================
# Skill 2: XML自動修正
# Phase 2の _process_delete_operation() を汎用化・独立化
# ============================================================
def fix_xml_structure(xml_config: str, translated_query: str = "") -> Dict[str, Any]:
    """
    XMLの構造を自動修正する（v7強化版）

    修正履歴:
    - v5: translated_query不問の確定的修正
    - v7: 修正④ <n> → <name> 正規化（Junos NETCONF正式タグ）
          修正⑤ 削除操作の子要素を <name> だけに強制クリーンアップ
                 （別AI指摘: operation="delete"はvlanタグに付与し子はnameのみ）

    Junos NETCONF 削除の正式形式:
      <vlan operation="delete"><name>VLAN名</name></vlan>
      ※ <name>以外の子要素が残るとJunosが拒否またはコンテナが残存する
    """
    if not xml_config:
        return {"fixed_xml": xml_config, "changes": ["No XML to fix"], "success": False}

    changes = []

    try:
        root = ET.fromstring(xml_config)

        # 修正①: <vlan> が直接 <configuration> 下 → <vlans> でラップ
        vlan_direct = root.find('vlan')
        if vlan_direct is not None:
            vlans_elem = ET.Element('vlans')
            root.remove(vlan_direct)
            vlans_elem.append(vlan_direct)
            root.append(vlans_elem)
            changes.append("Fixed: Added missing <vlans> wrapper around <vlan>")

        # 修正②: <operation>タグ形式 → 属性形式に変換（全vlan対象）
        for vlan in root.findall('.//vlan'):
            op_tag = vlan.find('operation')
            if op_tag is not None:
                op_value = (op_tag.text or '').strip()
                vlan.set('operation', op_value)
                vlan.remove(op_tag)
                changes.append(f"Fixed: Converted <operation>{op_value}</operation> to attribute")

        # 修正③④⑤: 削除操作の強制クリーンアップ（別AI指摘を統合）
        #   ③ <vlan-id> を除去
        #   ④ <n> → <name> に正規化（Junosは<name>が正式NETCONFタグ）
        #   ⑤ <name>以外の残存子要素を全除去（完全削除を保証）
        for vlan in root.findall('.//vlan[@operation="delete"]'):
            # ④ まず <n> を <name> に正規化してから処理
            n_tag = vlan.find('n')
            if n_tag is not None:
                name_tag = ET.Element('name')
                name_tag.text = n_tag.text
                idx = list(vlan).index(n_tag)
                vlan.remove(n_tag)
                vlan.insert(idx, name_tag)
                changes.append(f"Fixed: Renamed <n>'{name_tag.text}'</n> to <name> (Junos NETCONF standard)")

            # ⑤ <name> 以外の子要素を全て除去（<vlan-id>等）
            name_elem = vlan.find('name')
            vlan_name = name_elem.text if name_elem is not None else None
            for child in list(vlan):
                if child.tag != 'name':
                    vlan.remove(child)
                    changes.append(f"Fixed: Removed <{child.tag}> from delete operation (only <name> allowed)")

            # <name> が消えていた場合の防衛（念のため）
            if vlan.find('name') is None and vlan_name:
                name_tag = ET.SubElement(vlan, 'name')
                name_tag.text = vlan_name

        fixed_xml = ET.tostring(root, encoding='unicode')
        return {
            "fixed_xml": fixed_xml,
            "changes": changes if changes else ["No changes needed"],
            "success": True
        }

    except ET.ParseError as e:
        return {
            "fixed_xml": xml_config,
            "changes": [f"XML parse error - cannot fix: {str(e)}"],
            "success": False
        }

# ============================================================
# Skill 3: NETCONFデプロイ
# Phase 2の step4_deploy_config() を独立化
# ============================================================
def deploy_netconf_config(
    xml_config: str,
    device_ip: str,
    username: str,
    password: str,
    port: str = "830",
    comment: str = "AI Agent - Phase 3 Skill"
) -> Dict[str, Any]:
    """
    XML設定をJunosデバイスにNETCONFでデプロイする（v6修正版）

    修正点:
    - ConfigLoadError を個別捕捉
      severity: warning → rollback して no_changes 扱い
        例: "statement not found: SALES_VLAN"
            （削除対象が存在しない場合のJunos正常応答）
      severity: error   → failure として扱う

    Returns:
        {
            "status": "success" | "no_changes" | "failure" | "skipped",
            "diff": str,
            "message": str
        }
    """
    if not JUNOS_AVAILABLE:
        return {
            "status": "skipped",
            "diff": "",
            "message": "Junos modules not available"
        }

    try:
        with Device(host=device_ip, user=username, password=password, port=int(port)) as dev:
            cu = Config(dev)

            try:
                cu.load(xml_config, format="xml", merge=True)
            except ConfigLoadError as load_err:
                # severity: warning はデバイスの正常応答
                # 例: "statement not found" = 削除対象が存在しない
                # ★ v3: severity:error でも "statement not found" は
                #        Junos がデバイスによって severity を変えることがある。
                #        削除対象が存在しないだけなので no_changes として扱う。
                severity = getattr(load_err, 'severity', 'error')
                err_msg = getattr(load_err, 'message', str(load_err))
                is_not_found = "statement not found" in err_msg.lower()
                if severity == 'warning' or is_not_found:
                    cu.rollback()
                    return {
                        "status": "no_changes",
                        "diff": "",
                        "message": f"No changes applied (target not found): {err_msg}"
                    }
                else:
                    cu.rollback()
                    return {
                        "status": "failure",
                        "diff": "",
                        "message": f"ConfigLoadError (severity:{severity}): {err_msg}"
                    }

            diff = cu.diff()

            if diff:
                cu.commit(comment=comment)
                return {
                    "status": "success",
                    "diff": diff,
                    "message": f"Configuration deployed successfully to {device_ip}"
                }
            else:
                return {
                    "status": "no_changes",
                    "diff": "",
                    "message": "No configuration changes detected"
                }

    except Exception as e:
        return {
            "status": "failure",
            "diff": "",
            "message": f"Deployment failed: {str(e)}"
        }


# ============================================================
# 🎯 Skillオブジェクトの登録
# ============================================================
validate_xml_skill = Skill(
    name="validate_xml",
    description=(
        "Validate Junos XML configuration structure. "
        "Returns valid(bool), errors(list), warnings(list). "
        "Use this BEFORE deploying to catch structural issues."
    ),
    function=validate_xml_structure,
    parameters={
        "xml_config": {"type": "string", "description": "XML configuration string to validate"}
    }
)

fix_xml_skill = Skill(
    name="fix_xml",
    description=(
        "Auto-fix common Junos XML structural errors. "
        "Fixes: missing <vlans> wrapper, operation tag → attribute, removes <vlan-id> from delete ops. "
        "Returns fixed_xml(str), changes(list), success(bool)."
    ),
    function=fix_xml_structure,
    parameters={
        "xml_config": {"type": "string", "description": "XML configuration string to fix"},
        "translated_query": {"type": "string", "description": "English translation of user request (for context)"}
    }
)

deploy_skill = Skill(
    name="deploy_netconf",
    description=(
        "Deploy XML configuration to Junos device via NETCONF. "
        "Returns status(success/no_changes/failure/skipped), diff(str), message(str). "
        "Use ONLY after validate_xml confirms valid=True."
    ),
    function=deploy_netconf_config,
    parameters={
        "xml_config": {"type": "string", "description": "Valid XML configuration to deploy"},
        "device_ip": {"type": "string", "description": "Device IP address"},
        "username": {"type": "string", "description": "SSH username"},
        "password": {"type": "string", "description": "SSH password"},
        "port": {"type": "string", "description": "NETCONF port (default: 830)"}
    }
)

# ============================================================
# Skill 4: ロールバック（Phase 4継承）
# ============================================================

def rollback_config(
    device_ip: str,
    username: str,
    password: str,
    port: str = "830",
    mode: str = "candidate",
) -> Dict[str, Any]:
    """
    Junosデバイスの設定をロールバックする（Phase 4 新規）

    mode="candidate": candidate設定を破棄（rollback 0）
        → デプロイ失敗時。commitされていない変更を取り消す
    mode="rescue": rescue configurationへ復元してcommit
        → audit失敗時。最後の既知の正常状態に戻す
    """
    if not JUNOS_AVAILABLE:
        return {"status": "skipped", "mode": mode, "message": "Junos modules not available"}

    try:
        with Device(host=device_ip, user=username, password=password, port=int(port)) as dev:
            cu = Config(dev)
            if mode == "candidate":
                cu.rollback(0)
                return {
                    "status": "success",
                    "mode": "candidate",
                    "message": "Candidate configuration discarded (rollback 0)"
                }
            elif mode == "rescue":
                cu.rescue(action="reload")
                cu.commit(comment="Rollback to rescue configuration by AI Agent")
                return {
                    "status": "success",
                    "mode": "rescue",
                    "message": "Restored to rescue configuration and committed"
                }
            else:
                return {"status": "failure", "mode": mode, "message": f"Unknown mode: {mode}"}
    except Exception as e:
        return {"status": "failure", "mode": mode, "message": f"Rollback failed: {str(e)}"}


# ============================================================
# Skill 5: デプロイ後監査（Phase 4 NEW）
# ============================================================

def audit_deployment(
    xml_config: str,
    device_ip: str,
    username: str,
    password: str,
    port: str = "830",
) -> Dict[str, Any]:
    """
    デプロイ後に設定反映を確認する監査Skill（Phase 4 新規）

    NETCONFで show configuration vlans を取得し、
    デプロイしたXMLの内容が意図通り反映されているかを確認する。

    - operation="delete" → VLANが設定から消えていることを確認
    - operation="create" → VLANが設定に存在することを確認
    """
    if not JUNOS_AVAILABLE:
        return {"status": "skipped", "operation": "unknown",
                "vlan_name": "", "evidence": "", "message": "Junos modules not available"}

    import xml.etree.ElementTree as _ET

    try:
        root = _ET.fromstring(xml_config)
        vlan = root.find('.//vlan')
        if vlan is None:
            return {"status": "failure", "operation": "unknown", "vlan_name": "",
                    "evidence": "", "message": "No <vlan> element found in XML"}

        ### name_tag = vlan.find('name') or vlan.find('n')
        res = vlan.find('name')
        name_tag = res if res is not None else vlan.find('n')
        vlan_name = name_tag.text.strip() if name_tag is not None else ""
        op_attr = vlan.get('operation', '')
        operation = "delete" if op_attr == "delete" else "create"

    except _ET.ParseError as e:
        return {"status": "failure", "operation": "unknown", "vlan_name": "",
                "evidence": "", "message": f"XML parse error: {str(e)}"}

    try:
        with Device(host=device_ip, user=username, password=password, port=int(port)) as dev:
            result = dev.rpc.get_config(
                filter_xml='<configuration><vlans/></configuration>',
                options={'format': 'text'}
            )
            evidence = result.text if hasattr(result, 'text') else str(result)

            if operation == "delete":
                confirmed = vlan_name not in evidence
                return {
                    "status": "success" if confirmed else "failure",
                    "operation": "delete",
                    "vlan_name": vlan_name,
                    "evidence": evidence,
                    "message": (
                        f"✅ Confirmed: {vlan_name} deleted from configuration"
                        if confirmed else
                        f"❌ Audit failed: {vlan_name} still exists after delete"
                    )
                }
            else:
                confirmed = vlan_name in evidence
                return {
                    "status": "success" if confirmed else "failure",
                    "operation": "create",
                    "vlan_name": vlan_name,
                    "evidence": evidence,
                    "message": (
                        f"✅ Confirmed: {vlan_name} present in configuration"
                        if confirmed else
                        f"❌ Audit failed: {vlan_name} not found after create"
                    )
                }

    except Exception as e:
        return {"status": "failure", "operation": operation, "vlan_name": vlan_name,
                "evidence": "", "message": f"Audit error: {str(e)}"}


# ============================================================
# 🎯 Skillオブジェクトの登録（Phase 4: rollback/audit追加）
# ============================================================

rollback_skill = Skill(
    name="rollback",
    description=(
        "Rollback Junos device configuration. "
        "mode='candidate': discard uncommitted changes (rollback 0) — use after deploy failure. "
        "mode='rescue': restore to rescue configuration — use after audit failure. "
        "Returns status, mode, message."
    ),
    function=rollback_config,
    parameters={
        "device_ip": {"type": "string", "description": "Device IP address"},
        "username": {"type": "string", "description": "SSH username"},
        "password": {"type": "string", "description": "SSH password"},
        "port": {"type": "string", "description": "NETCONF port (default: 830)"},
        "mode": {"type": "string", "description": "'candidate' or 'rescue'"}
    }
)

audit_skill = Skill(
    name="audit",
    description=(
        "Verify deployment via NETCONF show configuration vlans. "
        "Confirms delete removed the VLAN, or create added it. "
        "Returns status, vlan_name, evidence(str), message."
    ),
    function=audit_deployment,
    parameters={
        "xml_config": {"type": "string", "description": "Deployed XML (determines VLAN name and operation)"},
        "device_ip": {"type": "string", "description": "Device IP address"},
        "username": {"type": "string", "description": "SSH username"},
        "password": {"type": "string", "description": "SSH password"},
        "port": {"type": "string", "description": "NETCONF port (default: 830)"}
    }
)

# ============================================================
# Skill 6: デバイスInventory取得（Phase 5 NEW）
# ============================================================

def get_device_inventory(
    device_ip: str,
    username: str,
    password: str,
    port: str = "830",
) -> Dict[str, Any]:
    """
    デバイスの現在のVLAN一覧をNETCONFで取得する（Phase 5 新規）

    Generator/Reviewerへの事前情報として渡すことで、
    「存在しないVLANを削除しようとする」ミスをLLMレベルで防ぐ。

    Returns:
        {
            "status": "success" | "failure" | "skipped",
            "vlans": [{"name": str, "vlan_id": str}, ...],
            "vlan_names": [str, ...],   # 名前リスト（プロンプト埋め込み用）
            "raw_config": str,          # show configuration vlansの生テキスト
            "message": str
        }
    """
    if not JUNOS_AVAILABLE:
        return {"status": "skipped", "vlans": [], "vlan_names": [],
                "raw_config": "", "message": "Junos modules not available"}

    try:
        with Device(host=device_ip, user=username, password=password, port=int(port)) as dev:
            result = dev.rpc.get_config(
                filter_xml='<configuration><vlans/></configuration>',
                options={'format': 'text'}
            )
            raw_config = result.text if hasattr(result, 'text') else str(result)

            # VLAN一覧をパース（2パス方式）
            # raw_configの構造:
            #   vlans {
            #       VLAN名 {        ← 4スペースインデント
            #           vlan-id N;
            #       }
            #   }
            vlans = []
            vlan_names = []
            import re as _re
            _lines = raw_config.split('\n')
            _i = 0
            while _i < len(_lines):
                _m = _re.match(r'^    (\S+)\s*\{', _lines[_i])
                if _m and _m.group(1) not in ('vlans',):
                    _name = _m.group(1)
                    _j = _i + 1
                    _vid = None
                    while _j < len(_lines) and '}' not in _lines[_j]:
                        _mv = _re.search(r'vlan-id\s+(\d+)', _lines[_j])
                        if _mv:
                            _vid = _mv.group(1)
                        _j += 1
                    if _vid:
                        vlans.append({"name": _name, "vlan_id": _vid})
                        vlan_names.append(_name)
                _i += 1

            return {
                "status": "success",
                "vlans": vlans,
                "vlan_names": vlan_names,
                "raw_config": raw_config,
                "message": f"Retrieved {len(vlans)} VLANs from {device_ip}"
            }

    except Exception as e:
        return {"status": "failure", "vlans": [], "vlan_names": [],
                "raw_config": "", "message": f"Inventory failed: {str(e)}"}


# ============================================================
# Skill 7: RAG検索をSkill化（Phase 5 NEW）
# ============================================================

def lookup_documentation(
    query: str,
    retriever=None,
    top_k: int = 3,
) -> Dict[str, Any]:
    """
    RAGドキュメント検索をSkill化（Phase 5 新規）

    Phase 1〜4では毎回強制実行していたRAG検索を、
    エージェントが「必要と判断した時だけ」呼べるSkillとして独立。

    GeneratorプロンプトにReAct判断基準を組み込むことで、
    LLMが自律的にこのSkillを要求するかどうかを判断する。

    Returns:
        {
            "status": "success" | "failure",
            "documents": [str, ...],
            "context": str,    # そのままプロンプトに埋め込める形式
            "message": str
        }
    """
    if retriever is None:
        return {"status": "failure", "documents": [], "context": "",
                "message": "Retriever not available"}
    try:
        docs = retriever.invoke(query)
        documents = [doc.page_content for doc in docs[:top_k]]
        context = "\n\n---\n\n".join(documents)
        return {
            "status": "success",
            "documents": documents,
            "context": context,
            "message": f"Found {len(documents)} documents for: {query}"
        }
    except Exception as e:
        return {"status": "failure", "documents": [], "context": "",
                "message": f"Lookup failed: {str(e)}"}


# ============================================================
# 🎯 Skillオブジェクトの登録（Phase 5: get_inventory/lookup_documentation追加）
# ============================================================

get_inventory_skill = Skill(
    name="get_inventory",
    description=(
        "Fetch current VLAN list from Junos device via NETCONF before deployment. "
        "Returns vlans(list), vlan_names(list), raw_config(str). "
        "Use to prevent operating on non-existent VLANs."
    ),
    function=get_device_inventory,
    parameters={
        "device_ip": {"type": "string", "description": "Device IP address"},
        "username": {"type": "string", "description": "SSH username"},
        "password": {"type": "string", "description": "SSH password"},
        "port": {"type": "string", "description": "NETCONF port (default: 830)"}
    }
)

lookup_documentation_skill = Skill(
    name="lookup_documentation",
    description=(
        "Search RAG knowledge base for Junos configuration documentation. "
        "Use when XML generation is uncertain or error occurs. "
        "Returns documents(list), context(str)."
    ),
    function=lookup_documentation,
    parameters={
        "query": {"type": "string", "description": "Search query for documentation"},
        "top_k": {"type": "integer", "description": "Number of documents to retrieve (default: 3)"}
    }
)

# Skill一覧
ALL_SKILLS = [
    validate_xml_skill, fix_xml_skill, deploy_skill,
    rollback_skill, audit_skill,
    get_inventory_skill, lookup_documentation_skill
]

print("✅ Skills定義完了")
print(f"   登録Skill数: {len(ALL_SKILLS)}")
for sk in ALL_SKILLS:
    print(f"   🛠️  {sk.name}: {sk.description[:60]}...")

✅ Skills定義完了
   登録Skill数: 7
   🛠️  validate_xml: Validate Junos XML configuration structure. Returns valid(bo...
   🛠️  fix_xml: Auto-fix common Junos XML structural errors. Fixes: missing ...
   🛠️  deploy_netconf: Deploy XML configuration to Junos device via NETCONF. Return...
   🛠️  rollback: Rollback Junos device configuration. mode='candidate': disca...
   🛠️  audit: Verify deployment via NETCONF show configuration vlans. Conf...
   🛠️  get_inventory: Fetch current VLAN list from Junos device via NETCONF before...
   🛠️  lookup_documentation: Search RAG knowledge base for Junos configuration documentat...


## 🔄 Phase 6 ワークフロー

```
[Orchestrator]
  ↓
  [Step 1] task_decomposer     ← 自然言語 → タスクリスト JSON
  [Step 2] dependency_resolver ← DAG生成・トポロジカルソート
  [Step 3] Worker へ順次ディスパッチ（依存順）
           ↓ 各 Worker (Phase 5 継承・変更なし)
           get_inventory → translate → Generator(ReAct) → validate → fix → deploy → audit → rollback
  [Step 4] result_aggregator   ← 全Task結果を集約・最終レポート生成

失敗時: いずれかの Task が失敗 → 残タスクを中断 → 状況報告
```

### Reviewer の修正（Phase 6 で解決）
- **Before**: Junos 仕様ルールをプロンプトに含む → 正しい XML を誤 REJECT するリスク
- **After**: 「ユーザー意図との整合性チェック」のみ。Junos 仕様判断は validate_xml Skill に完全移譲


In [6]:
class NetconfRagWorkflowPhase3:
    """
    NETCONF RAG Workflow Phase 3
    Skill化ハイブリッド実装 (v2 - ルールベースSkillループ)

    Phase 2からの変更点:
    - validate / fix / deploy を Skill として登録
    - Skillループをルールベースの確定的制御に変更
      （SkillAgentによるLLM判断は不安定なため廃止）
    - Skill実行ログを skill_execution_log で追跡
    """

    def __init__(
        self,
        retriever,         # ★ v2: api_key 引数を削除（make_client() がグローバル変数を使用）
        llm,
        skills: List[Skill] = None,
        max_retries: int = 3,
        max_review_rounds: int = 2
    ):
        # ★ v2: self.api_key 削除（OpenAIChatClient は make_client() 経由でGROQ_API_KEYを使用）
        self.retriever = retriever
        self.llm = llm
        self.max_retries = max_retries
        self.max_review_rounds = max_review_rounds
        self.logs = []
        self.conversation_history: List[Message] = []
        self.skill_execution_log: List[Dict] = []

        self.skills: Dict[str, Skill] = {}
        for sk in (skills or ALL_SKILLS):
            self.skills[sk.name] = sk

        self._initialize_agents()

    def _initialize_agents(self):
        """2つのエージェントを初期化（Generator・Reviewer）"""

        # ★ v2: GroqChatClient → make_client() (OpenAIChatClient)
        generator_client = make_client(DEFAULT_MODEL)
        generator_instructions = """
You are a dedicated JUNOS NETCONF XML generator.
Your ONLY task is to produce valid JUNOS XML based on the provided Context and Request.

[CRITICAL RULES]
1. OUTPUT ONLY RAW XML.
2. DO NOT include any introductory text, explanations, or "Here is the XML".
3. DO NOT use markdown code blocks (no ```xml).
4. Start immediately with <configuration> and end with </configuration>.
5. If you cannot generate XML, output ONLY an empty <configuration/> tag.
6. Ignore any Python code snippets in the Context; only use the configuration logic.

[CRITICAL TAG RULE]
- VLAN name tag MUST be <name>...</name>  ← ONLY correct tag
- Do NOT use <name> tag for VLAN name

[BAD EXAMPLE]
<vlan><operation>delete</operation></vlan>

[GOOD EXAMPLE - DELETE]
<vlan operation="delete"><name>VLAN_NAME</name></vlan>

[GOOD EXAMPLE - CREATE]
<vlan><name>VLAN_NAME</name><vlan-id>70</vlan-id></vlan>

[STRUCTURE EXAMPLE]
<configuration>
    <vlans>
        <vlan operation="delete">
            <name>SALES_VLAN</name>
        </vlan>
    </vlans>
</configuration>
"""
        self.xml_generator = Agent(
            name="XMLGenerator",
            client=generator_client,
            instructions=generator_instructions
        )

        # ★ v2: GroqChatClient → make_client() (OpenAIChatClient)
        reviewer_client = make_client(DEFAULT_MODEL)
        reviewer_instructions = """
You are a JUNOS XML configuration reviewer and syntax validator.

[YOUR TASK]
Review the provided JUNOS XML and check for structural correctness.

[CRITICAL STRUCTURE RULES]
1. Hierarchy Validation:
   - VLANs MUST be under <vlans> tag: <configuration><vlans><vlan>
   - Interfaces MUST be under <interfaces> tag
   - NEVER accept <vlan> directly under <configuration>

2. Delete Operation Format:
   - MUST use: <vlan operation="delete"><name>vlan-name</name></vlan>
   - MUST have <name> tag with VLAN name
   - Must NOT have <vlan-id> tag
   - NOTE: <name> is the ONLY correct tag. Do NOT require or accept <name> tag.

3. Create Operation Format:
   - MUST have both <name> and <vlan-id> tags

[IMPORTANT: TAG NAMING]
- The correct VLAN name tag is <name>, NOT <name>
- APPROVE any XML that uses <name> tag correctly
- REJECT only if <name> tag is completely missing or wrong structure

[OUTPUT FORMAT - MANDATORY]
APPROVE: Configuration structure is correct.
IMPROVE: <specific issue>
REJECT: <critical structural error>

DO NOT generate new XML. ONLY review and respond with APPROVE, IMPROVE, or REJECT.
"""
        self.xml_reviewer = Agent(
            name="XMLReviewer",
            client=reviewer_client,
            instructions=reviewer_instructions
        )

        self.log("✅ 2つのエージェント初期化完了")
        self.log("   🤖 XMLGenerator  - XML生成")
        self.log("   👁️  XMLReviewer   - レビュー・改善提案")
        self.log(f"   📋 登録Skills: {', '.join(self.skills.keys())} (ルールベース実行)")

    def log(self, message: str):
        self.logs.append(message)
        print(message)

    def add_message(self, role: str, text: str):
        self.conversation_history.append(Message(role=role, text=text))

    def _extract_response_text(self, response) -> str:
        if hasattr(response, 'text'):
            return str(response.text)
        if hasattr(response, 'messages') and response.messages:
            for msg in response.messages:
                if hasattr(msg, 'text'):
                    return str(msg.text)
        return str(response)

    def _extract_xml(self, response: str) -> str:
        xml_match = re.search(r'```xml\s*(<configuration>.*?</configuration>)\s*```', response, re.DOTALL)
        if xml_match:
            return xml_match.group(1).strip()
        xml_match = re.search(r'(<configuration>.*?</configuration>)', response, re.DOTALL)
        if xml_match:
            return xml_match.group(1).strip()
        return ""

    def _run_skill(self, skill_name: str, **kwargs) -> Any:
        # lookup_documentationスキルにretrieverを自動注入
        if skill_name == "lookup_documentation" and "retriever" not in kwargs:
            kwargs["retriever"] = self.retriever
        """
        Skillを実行し、実行ログを記録する

        Phase 3の本質: Skillは「明示的に登録された能力」なので
        呼び出し側がルールベースで確定的に実行する
        """
        if skill_name not in self.skills:
            self.log(f"  ❌ [Skill] 未知のSkill: {skill_name}")
            return None

        skill = self.skills[skill_name]
        self.log(f"  🛠️  [Skill:{skill_name}] 実行中...")

        try:
            result = skill.execute(**kwargs)
            log_entry = {
                "timestamp": datetime.now().isoformat(),
                "skill": skill_name,
                "params": {k: (v[:50] + "...") if isinstance(v, str) and len(v) > 50 else v
                           for k, v in kwargs.items()},
                "result_summary": str(result)[:200]
            }
            self.skill_execution_log.append(log_entry)
            self.log(f"  ✅ [Skill:{skill_name}] 完了")
            return result
        except Exception as e:
            self.log(f"  ❌ [Skill:{skill_name}] エラー: {e}")
            return None

    async def _run_skill_loop(
        self,
        xml_config: str,
        translated_query: str,
        device_info: Optional[Dict] = None,
        deploy: bool = False
    ) -> Dict[str, Any]:
        """
        Skillループ実行（ルールベース確定的制御）

        フロー: validate → (失敗なら fix → re-validate) → deploy

        v1との違い:
        - SkillAgentによるLLM判断を廃止（不安定だったため）
        - validate → fix → deploy の順序をコードで確定的に制御
        - Skillオブジェクトを使う点（明示的登録・実行ログ）は同じ
        """
        self.log("\n" + "="*70)
        self.log("🛠️  Skillループ開始 [Phase 3 v2 - ルールベース]")
        self.log("="*70)

        current_xml = xml_config
        skill_steps = []
        max_fix_attempts = 3  # 修正ループ上限（暴走防止）

        # ── ステップA: validate_xml Skill ──
        self.log("\n  📋 [Step A] validate_xml Skill 実行")
        skill_steps.append("validate")
        val_result = self._run_skill("validate_xml", xml_config=current_xml)

        if val_result is None:
            return {"final_xml": current_xml, "valid": False,
                    "deployment_status": {"status": "skipped", "diff": "", "message": "validate_xml Skill failed"},
                    "skill_steps": skill_steps}

        self.log(f"  📋 検証結果: valid={val_result['valid']}")
        if val_result['errors']:
            self.log(f"  ❌ エラー: {val_result['errors']}")
        if val_result['warnings']:
            self.log(f"  ⚠️  警告: {val_result['warnings']}")

        # ── ステップB: エラーがあれば fix_xml Skill → 再validate ──
        if not val_result['valid']:
            for fix_attempt in range(max_fix_attempts):
                self.log(f"\n  🔧 [Step B] fix_xml Skill 実行 (試行 {fix_attempt + 1}/{max_fix_attempts})")
                skill_steps.append("fix")

                fix_result = self._run_skill(
                    "fix_xml",
                    xml_config=current_xml,
                    translated_query=translated_query
                )

                if not fix_result or not fix_result['success']:
                    self.log("  ❌ fix_xml Skill 失敗")
                    break

                current_xml = fix_result['fixed_xml']
                self.log(f"  🔧 修正内容: {fix_result['changes']}")
                self.add_message("system", f"XML auto-fixed by fix_xml Skill: {fix_result['changes']}")

                # 修正後に再検証
                self.log("  📋 [Step B] fix後 再validate")
                skill_steps.append("re-validate")
                val_result = self._run_skill("validate_xml", xml_config=current_xml)

                if val_result and val_result['valid']:
                    self.log("  ✅ fix後 検証通過")
                    break
                elif val_result:
                    self.log(f"  ❌ fix後もエラー: {val_result['errors']}")
                else:
                    break

        # ── ステップC: validate通過確認 ──
        if not val_result or not val_result['valid']:
            self.log("  ❌ 最終的に検証失敗")
            return {
                "final_xml": current_xml,
                "valid": False,
                "deployment_status": {"status": "skipped", "diff": "", "message": "Validation failed after fix attempts"},
                "skill_steps": skill_steps
            }

        self.log("  ✅ XML検証通過 → デプロイ判断へ")

        # ── ステップD: deploy_netconf Skill ──
        if deploy and device_info:
            self.log("\n  📡 [Step D] deploy_netconf Skill 実行")
            skill_steps.append("deploy")

            deploy_result = self._run_skill(
                "deploy_netconf",
                xml_config=current_xml,
                device_ip=device_info['ip'],
                username=device_info['username'],
                password=device_info['password'],
                port=device_info.get('port', '830')
            )

            if not deploy_result or deploy_result['status'] == "failure":
                # デプロイ失敗 → candidate破棄（rollback 0）
                err_msg = deploy_result.get('message', 'unknown') if deploy_result else 'Skill error'
                self.log(f"  ❌ デプロイ失敗: {err_msg}")
                self.log("\n  🔙 [Step F] rollback Skill 実行 (mode=candidate / デプロイ失敗)")
                skill_steps.append("rollback(candidate)")
                rb_result = self._run_skill(
                    "rollback",
                    device_ip=device_info['ip'],
                    username=device_info['username'],
                    password=device_info['password'],
                    port=device_info.get('port', '830'),
                    mode="candidate"
                )
                self.log(f"  🔙 ロールバック: {rb_result.get('message','') if rb_result else 'error'}")
                return {
                    "final_xml": current_xml, "valid": True,
                    "deployment_status": deploy_result or {"status": "failure", "diff": "", "message": "deploy_netconf Skill failed"},
                    "audit_status": None, "rollback_status": rb_result,
                    "skill_steps": skill_steps
                }

            self.log(f"  📡 デプロイ結果: {deploy_result['status']}")
            if deploy_result.get('diff'):
                self.log(f"  📄 差分:\n{deploy_result['diff']}")

            # ── ステップE: audit Skill（Phase 4 NEW）──
            self.log("\n  🔍 [Step E] audit Skill 実行")
            skill_steps.append("audit")
            audit_result = self._run_skill(
                "audit",
                xml_config=current_xml,
                device_ip=device_info['ip'],
                username=device_info['username'],
                password=device_info['password'],
                port=device_info.get('port', '830')
            )

            if audit_result:
                self.log(f"  🔍 audit結果: {audit_result['status']}")
                self.log(f"  📋 {audit_result['message']}")
                if audit_result.get('evidence'):
                    lines = audit_result['evidence'].strip().split('\n')
                    self.log(f"  📄 エビデンス (show configuration vlans):")
                    for line in lines[:15]:
                        self.log(f"       {line}")

            if not audit_result or audit_result['status'] == "failure":
                # audit失敗 → rescue configurationへ復元
                self.log("\n  🔙 [Step F] rollback Skill 実行 (mode=rescue / audit失敗)")
                skill_steps.append("rollback(rescue)")
                rb_result = self._run_skill(
                    "rollback",
                    device_ip=device_info['ip'],
                    username=device_info['username'],
                    password=device_info['password'],
                    port=device_info.get('port', '830'),
                    mode="rescue"
                )
                self.log(f"  🔙 ロールバック: {rb_result.get('message','') if rb_result else 'error'}")
                return {
                    "final_xml": current_xml, "valid": True,
                    "deployment_status": deploy_result,
                    "audit_status": audit_result, "rollback_status": rb_result,
                    "skill_steps": skill_steps
                }

            return {
                "final_xml": current_xml, "valid": True,
                "deployment_status": deploy_result,
                "audit_status": audit_result, "rollback_status": None,
                "skill_steps": skill_steps
            }

        else:
            reason = "deploy=False" if not deploy else "device_info missing"
            self.log(f"  ⏭️  デプロイスキップ ({reason})")
            return {
                "final_xml": current_xml,
                "valid": True,
                "deployment_status": {"status": "skipped", "diff": "", "message": reason},
                "skill_steps": skill_steps
            }

    async def step1_translate_query(self, user_query: str) -> str:
        self.log("\n" + "="*70)
        self.log("ステップ1: 質問を技術的な英語に変換中...")
        self.log("="*70)

        # ★ Phase 6: Orchestrator からは既に英語クエリが渡されるためバイパス
        # ASCII文字が80%以上 → 英語と判定してそのままパススルー
        ascii_ratio = sum(1 for c in user_query if ord(c) < 128) / max(len(user_query), 1)
        if ascii_ratio >= 0.8:
            self.log(f"  ℹ️  英語クエリを検出 → 翻訳スキップ (ascii_ratio={ascii_ratio:.2f})")
            self.log(f"変換結果: {user_query}")
            self.add_message("user", user_query)
            self.add_message("system", f"Translation: (skipped - already English) {user_query}")
            return user_query

        translation_prompt = (
            f"Convert the following Japanese network configuration request into a precise English technical command: "
            f"'{user_query}'. Output only the command text."
        )
        response = self.llm.invoke(translation_prompt)
        translated_text = response.content.strip()
        self.log(f"変換結果: {translated_text}")
        self.add_message("user", user_query)
        self.add_message("system", f"Translation: {translated_text}")
        return translated_text

    async def step2_retrieve_information(self, translated_query: str) -> List[str]:
        self.log("\n" + "="*70)
        self.log("ステップ2: RAG検索中...")
        self.log("="*70)
        if not self.retriever:
            self.log("⚠️ Retrieverが利用できません")
            return []
        docs = self.retriever.invoke(translated_query)
        documents_content = [doc.page_content for doc in docs]
        self.log(f"検索結果: {len(documents_content)}件")
        self.add_message("system", f"Retrieved {len(documents_content)} documents")
        return documents_content

    async def step3_generate_and_review_xml(
        self,
        translated_query: str,
        retrieved_docs: List[str],
        inventory_info: Dict = None
    ) -> tuple:
        self.log("\n" + "="*70)
        self.log("ステップ3: マルチエージェント XML生成・レビュー")
        self.log("="*70)
        context = "\n\n---\n\n".join(retrieved_docs)

        for attempt in range(self.max_retries):
            self.log(f"\n🔄 生成試行 {attempt + 1}/{self.max_retries}")
            # Phase 5: Inventory情報をプロンプトに埋め込む
            inventory_section = ""
            if inventory_info and inventory_info.get('status') == 'success':
                vlan_list = inventory_info.get('raw_config', '').strip()
                vlan_names = inventory_info.get('vlan_names', [])
                inventory_section = f"""
### [IMPORTANT] Current Device State (get_inventory result):
Existing VLANs on device: {vlan_names if vlan_names else '(no VLANs configured)'}
Raw configuration:
{vlan_list if vlan_list.strip() else '(empty)'}

CRITICAL CHECK: Before generating XML, verify the target VLAN exists for delete/modify operations.
If the target VLAN is NOT in the existing list, still generate the XML but note the discrepancy.
"""

            gen_prompt = f"""
You are a Junos network engineer. Generate ONLY the JUNOS XML configuration.

[BAD EXAMPLE - WRONG]
<vlan><operation>delete</operation></vlan>   ← operation must be an attribute, NOT a child tag

[GOOD EXAMPLE - CORRECT]
<vlan operation="delete"><name>VLAN_NAME</name></vlan>

[STRICT XML RULES]
- Output ONLY <configuration>...</configuration>. No explanations. No markdown.
- DELETE/REMOVE → operation="delete" attribute on <vlan> tag
- CREATE/ADD → no operation attribute, include <name> and <vlan-id>
- Use <name> tag (NOT <name>) for VLAN name
- DELETE: include ONLY <name> tag, NO <vlan-id>
{inventory_section}
### Documentation Context:
{context}

### Request:
{translated_query}

Generate JUNOS XML now. ONLY XML. NO EXPLANATIONS.
"""
            self.log("  🤖 [Generator] XML生成中...")
            gen_response = await self.xml_generator.run(gen_prompt)
            raw_xml = self._extract_response_text(gen_response)
            generated_xml = self._extract_xml(raw_xml)

            if not generated_xml:
                self.log("  ❌ [Generator] XML抽出失敗")
                continue
            try:
                ET.fromstring(generated_xml)
            except ET.ParseError as e:
                self.log(f"  ❌ XMLパースエラー: {e}")
                continue

            self.log("  ✅ [Generator] XML生成成功")
            self.add_message("assistant", f"[Generator] Generated XML (attempt {attempt + 1})")

            for review_round in range(self.max_review_rounds):
                self.log(f"\n  👁️  [Reviewer] レビューラウンド {review_round + 1}/{self.max_review_rounds}")
                inventory_check = ""
                if inventory_info and inventory_info.get('status') == 'success':
                    vlan_names = inventory_info.get('vlan_names', [])
                    inventory_check = f"""
Existing VLANs on device: {vlan_names if vlan_names else '(none)'}
Verify: does the XML target a VLAN that exists (for delete) or not yet exist (for create)?
"""
                review_prompt = f"""
Review this JUNOS XML for Junos NETCONF correctness:
{generated_xml}

User requirement: {translated_query}
{inventory_check}
[CRITICAL JUNOS DELETE RULES]
When DELETING a VLAN, Junos NETCONF requires ONLY the VLAN name - NO vlan-id:
  CORRECT: <vlan operation="delete"><name>VLAN_NAME</name></vlan>
  WRONG:   <vlan operation="delete"><name>VLAN_NAME</name><vlan-id>70</vlan-id></vlan>

If user says "delete VLAN ID 70", that means identify which VLAN to delete.
The XML must NOT include <vlan-id> in a delete operation. This is Junos spec.
A delete XML with only <name> tag is CORRECT for Junos - APPROVE it.

Check:
1. operation="delete" is an XML attribute on <vlan>, not a child tag
2. DELETE: contains ONLY <name> or <name> tag - NO <vlan-id> (this is CORRECT per Junos)
3. CREATE: contains both <name>/<name> and <vlan-id>
4. Uses <name> or <name> tag for VLAN name

IMPORTANT: Your response MUST start with exactly one of: APPROVE, IMPROVE, or REJECT.
First word must be the verdict. Format: "APPROVE." or "REJECT. <reason>"
"""
                review_response = await self.xml_reviewer.run(review_prompt)
                review_text = self._extract_response_text(review_response)
                self.log(f"  📋 [Reviewer] {review_text[:150]}")
                self.add_message("assistant", f"[Reviewer] {review_text[:100]}")

                if "APPROVE" in review_text.upper():
                    self.log("  ✅ [Reviewer] APPROVED")
                    return generated_xml, True
                elif "IMPROVE" in review_text.upper() and review_round < self.max_review_rounds - 1:
                    self.log("  🔧 [Reviewer] IMPROVE")
                    improvement_suggestion = review_text.split("IMPROVE:", 1)[-1].strip() if "IMPROVE:" in review_text else review_text
                    improve_prompt = f"""
[STRICT RULES]
- Output ONLY <configuration>...</configuration>.
- DELETE/REMOVE → operation="delete" attribute

Context: {context}
Request: {translated_query}
Previous XML: {generated_xml}
Reviewer Feedback: {improvement_suggestion}

Improve the XML now.
"""
                    improve_response = await self.xml_generator.run(improve_prompt)
                    improved_raw = self._extract_response_text(improve_response)
                    improved_xml = self._extract_xml(improved_raw)
                    if improved_xml:
                        try:
                            ET.fromstring(improved_xml)
                            generated_xml = improved_xml
                            self.log("  ✅ [Generator] 改善版生成成功")
                            continue
                        except ET.ParseError:
                            pass
                    self.log("  ⚠️ [Generator] 改善失敗 - 元のXMLで継続")
                    return generated_xml, True
                elif "REJECT" in review_text.upper():
                    self.log("  ❌ [Reviewer] REJECTED")
                    break
                else:
                    self.log("  ⚠️ [Reviewer] 不明応答 - 承認扱い")
                    return generated_xml, True

        self.log(f"\n⚠️ {self.max_retries}回の試行後もXML生成失敗")
        return "", False

    async def run(
        self,
        user_query: str,
        device_ip: str = None,
        username: str = None,
        password: str = None,
        port: str = "830",
        deploy: bool = False
    ) -> Dict[str, Any]:
        self.logs = []
        self.conversation_history = []
        self.skill_execution_log = []

        self.log("\n" + "="*70)
        self.log("🚀 Phase 5: Inventory & RAG Skill化ワークフロー（ReAct型）")
        self.log(f"🔒 暴走防止: 生成{self.max_retries}回 × レビュー{self.max_review_rounds}ラウンド")
        self.log(f"🛠️  登録Skills: {', '.join(self.skills.keys())}")
        self.log("="*70)
        self.log(f"入力: {user_query}")

        result = {
            'user_query': user_query,
            'translated_query': '',
            'retrieved_documents': [],
            'generated_xml': '',
            'final_xml': '',
            'validation_status': False,
            'deployment_status': {},
            'skill_steps': [],
            'skill_execution_log': [],
            'conversation_history': []
        }

        try:
            result['translated_query'] = await self.step1_translate_query(user_query)

            # ── Phase 5 Step 0: get_inventory（デプロイ前にデバイス状態を把握）──
            inventory_info = None
            if all([device_ip, username, password]):
                self.log("\n" + "="*70)
                self.log("ステップ0: デバイスInventory取得 [Phase 5 NEW]")
                self.log("="*70)
                inv_result = self._run_skill(
                    "get_inventory",
                    device_ip=device_ip,
                    username=username,
                    password=password,
                    port=port
                )
                if inv_result and inv_result['status'] == 'success':
                    inventory_info = inv_result
                    self.log(f"📋 現在のVLAN一覧: {inv_result['vlan_names'] if inv_result['vlan_names'] else '(空)'}")
                    self.log(f"   {inv_result['message']}")
                    result['inventory'] = inv_result
                else:
                    msg = inv_result.get('message','') if inv_result else 'Skill error'
                    self.log(f"⚠️ Inventory取得失敗（続行）: {msg}")

            # ── RAGはSkill化：最初の1回は常に実行し、contextとしてGeneratorに渡す ──
            # （Phase 5の本質: Generatorがさらに追加検索が必要か判断できるよう基礎contextを提供）
            result['retrieved_documents'] = await self.step2_retrieve_information(result['translated_query'])

            raw_xml, review_passed = await self.step3_generate_and_review_xml(
                result['translated_query'],
                result['retrieved_documents'],
                inventory_info=inventory_info
            )
            result['generated_xml'] = raw_xml

            if not review_passed or not raw_xml:
                self.log("\n❌ XML生成失敗 - ワークフロー中断")
                result['validation_status'] = False
                return result

            device_info = None
            if deploy and all([device_ip, username, password]):
                device_info = {
                    'ip': device_ip,
                    'username': username,
                    'password': password,
                    'port': port
                }
            elif deploy:
                self.log("⚠️ デバイス情報不足 - デプロイスキップ")

            skill_result = await self._run_skill_loop(
                xml_config=raw_xml,
                translated_query=result['translated_query'],
                device_info=device_info,
                deploy=deploy
            )

            result['final_xml'] = skill_result['final_xml']
            result['validation_status'] = skill_result['valid']
            result['deployment_status'] = skill_result['deployment_status']
            result['audit_status']    = skill_result.get('audit_status')    # ★ fix: 転写漏れ
            result['rollback_status'] = skill_result.get('rollback_status') # ★ fix: 転写漏れ
            result['skill_steps'] = skill_result['skill_steps']
            result['skill_execution_log'] = self.skill_execution_log
            result['conversation_history'] = [
                {"role": msg.role, "text": msg.text}
                for msg in self.conversation_history
            ]

            self.log("\n" + "="*70)
            self.log("✅ ワークフロー完了")
            self.log(f"   🛠️  Skillステップ: {' → '.join(skill_result['skill_steps'])}")
            self.log("="*70)

        except Exception as e:
            self.log(f"\n❌ エラー: {e}")
            import traceback
            traceback.print_exc()
            result['error'] = str(e)

        return result

print("✅ NetconfRagWorkflowPhase3 定義完了 (v2 - ルールベースSkillループ)")


✅ NetconfRagWorkflowPhase3 定義完了 (v2 - ルールベースSkillループ)


## ⚙️ 設定

In [7]:
# デバイス設定
DEVICE_CONFIG = {
    'ip': '192.0.2.1'  # Replace with your device IP,
    'port': '830',
    'username': 'admin',
    'password': 'YOUR_PASSWORD_HERE'
}

# テストクエリ（Phase 2 と同じクエリで動作確認）
###USER_QUERY = "インターフェイス 'et-0/0/0' に説明を設定します。その説明は 'uplink to core' とします"
USER_QUERY = "'SALES_VLAN' の 'VLAN ID 70'を追加してください。"
###USER_QUERY = "'SALES_VLAN' の 'VLAN ID 70'を削除してください。"

DEPLOY_TO_DEVICE = True    # True で実機投入、False で試験のみ
MAX_RETRIES = 3
MAX_REVIEW_ROUNDS = 2

print("✅ 設定完了")
print(f"   クエリ: {USER_QUERY}")
print(f"   デプロイ: {DEPLOY_TO_DEVICE}")

✅ 設定完了
   クエリ: 'SALES_VLAN' の 'VLAN ID 70'を追加してください。
   デプロイ: True


## 🚀 Phase 3 ワークフロー実行

In [8]:
# Phase 3 ワークフロー実行
# ★ v2: api_key 引数を削除（make_client() がGROQ_API_KEYを直接参照）
workflow = NetconfRagWorkflowPhase3(
    retriever=retriever,
    llm=llm,
    skills=ALL_SKILLS,       # 🆕 Skills を明示的に渡す
    max_retries=MAX_RETRIES,
    max_review_rounds=MAX_REVIEW_ROUNDS
)

result = await workflow.run(
    user_query=USER_QUERY,
    device_ip=DEVICE_CONFIG['ip'],
    username=DEVICE_CONFIG['username'],
    password=DEVICE_CONFIG['password'],
    port=DEVICE_CONFIG['port'],
    deploy=DEPLOY_TO_DEVICE
)

print("\n✅ 実行完了")

✅ 2つのエージェント初期化完了
   🤖 XMLGenerator  - XML生成
   👁️  XMLReviewer   - レビュー・改善提案
   📋 登録Skills: validate_xml, fix_xml, deploy_netconf, rollback, audit, get_inventory, lookup_documentation (ルールベース実行)

🚀 Phase 5: Inventory & RAG Skill化ワークフロー（ReAct型）
🔒 暴走防止: 生成3回 × レビュー2ラウンド
🛠️  登録Skills: validate_xml, fix_xml, deploy_netconf, rollback, audit, get_inventory, lookup_documentation
入力: 'SALES_VLAN' の 'VLAN ID 70'を追加してください。

ステップ1: 質問を技術的な英語に変換中...
変換結果: Add VLAN ID 70 to SALES_VLAN.

ステップ0: デバイスInventory取得 [Phase 5 NEW]
  🛠️  [Skill:get_inventory] 実行中...
  ✅ [Skill:get_inventory] 完了
📋 現在のVLAN一覧: ['MGMT_VLAN']
   Retrieved 1 VLANs from 172.20.100.21

ステップ2: RAG検索中...
検索結果: 5件

ステップ3: マルチエージェント XML生成・レビュー

🔄 生成試行 1/3
  🤖 [Generator] XML生成中...
  ✅ [Generator] XML生成成功

  👁️  [Reviewer] レビューラウンド 1/2
  📋 [Reviewer] APPROVE.
  ✅ [Reviewer] APPROVED

🛠️  Skillループ開始 [Phase 3 v2 - ルールベース]

  📋 [Step A] validate_xml Skill 実行
  🛠️  [Skill:validate_xml] 実行中...
  ✅ [Skill:validate_xml] 完了
  📋 検証結果: valid=True

## 📊 実行結果（Phase 4）

`deploy → audit → rollback(必要時)` の結果を表示します。


In [9]:
print("\n" + "="*70)
print("📊 Phase 3 実行結果")
print("="*70)

print(f"\n1️⃣  入力: {result['user_query']}")
print(f"2️⃣  翻訳: {result['translated_query']}")
print(f"3️⃣  RAG: {len(result['retrieved_documents'])}件")
print(f"4️⃣  検証: {'✅ 成功' if result['validation_status'] else '❌ 失敗'}")

if result.get('skill_steps'):
    print(f"5️⃣  Skillステップ: {' → '.join(result['skill_steps'])}")

if result.get('deployment_status'):
    status = result['deployment_status'].get('status', 'N/A')
    print(f"6️⃣  デプロイ: {status}")

audit = result.get('audit_status')
rollback = result.get('rollback_status')
if audit:
    print(f"7️⃣  Audit:    {audit.get('status','N/A')} - {audit.get('message','')}")
if rollback:
    print(f"8️⃣  Rollback: {rollback.get('status','N/A')} [{rollback.get('mode','')}] - {rollback.get('message','')}")

print("\n" + "="*70)
print("📄 生成XML (Generator出力)")
print("="*70)

def pretty_print_xml(xml_str):
    try:
        dom = minidom.parseString(xml_str)
        pretty = dom.toprettyxml(indent="  ")
        pretty = "\n".join([line for line in pretty.split("\n") if line.strip()])
        print(pretty)
    except:
        print(xml_str)

if result['generated_xml']:
    pretty_print_xml(result['generated_xml'])
else:
    print("⚠️ XML未生成")

print("\n" + "="*70)
print("📄 最終XML (Skill処理後)")
print("="*70)

if result.get('final_xml') and result['final_xml'] != result['generated_xml']:
    pretty_print_xml(result['final_xml'])
    print("\n  ↑ Skill修正が適用されました")
elif result.get('final_xml'):
    print("  （生成XMLと同一 - 修正不要）")

print("\n" + "="*70)


📊 Phase 3 実行結果

1️⃣  入力: 'SALES_VLAN' の 'VLAN ID 70'を追加してください。
2️⃣  翻訳: Add VLAN ID 70 to SALES_VLAN.
3️⃣  RAG: 5件
4️⃣  検証: ✅ 成功
5️⃣  Skillステップ: validate → deploy → audit
6️⃣  デプロイ: success
7️⃣  Audit:    success - ✅ Confirmed: SALES_VLAN present in configuration

📄 生成XML (Generator出力)
<?xml version="1.0" ?>
<configuration>
  <vlans>
    <vlan>
      <name>SALES_VLAN</name>
      <vlan-id>70</vlan-id>
    </vlan>
  </vlans>
</configuration>

📄 最終XML (Skill処理後)
  （生成XMLと同一 - 修正不要）



## 🛠️ Skill実行ログ（Phase 3 新規）

In [10]:
print("\n" + "="*70)
print("🛠️  Skill実行ログ [Phase 3]")
print("="*70)

if result.get('skill_execution_log'):
    for i, log_entry in enumerate(result['skill_execution_log'], 1):
        print(f"\n{i}. 🛠️  Skill: {log_entry['skill']}")
        print(f"   ⏰ 時刻: {log_entry['timestamp']}")
        print(f"   📥 パラメータ: {log_entry['params']}")
        print(f"   📤 結果: {log_entry['result_summary']}")
else:
    print("  （Skill実行なし）")

print("\n" + "="*70)
print("🔍 Audit / Rollback ログ [Phase 4]")
print("="*70)
audit = result.get('audit_status')
rollback = result.get('rollback_status')
if audit:
    print(f"\n🔍 Audit Skill:")
    print(f"   対象VLAN : {audit.get('vlan_name','')}")
    print(f"   操作     : {audit.get('operation','')}")
    print(f"   ステータス: {audit.get('status','')}")
    print(f"   メッセージ: {audit.get('message','')}")
    if audit.get('evidence'):
        print("   エビデンス (show configuration vlans):")
        for line in audit['evidence'].strip().split('\n')[:20]:
            print(f"     {line}")
else:
    print("   Audit: 未実行")
if rollback:
    print(f"\n🔙 Rollback Skill:")
    print(f"   モード    : {rollback.get('mode','')}")
    print(f"   ステータス: {rollback.get('status','')}")
    print(f"   メッセージ: {rollback.get('message','')}")
else:
    print("\n🔙 Rollback: 実行不要（正常完了）")

print("\n" + "="*70)
print("💬 エージェント対話履歴")
print("="*70)

for i, msg in enumerate(result['conversation_history'], 1):
    role_emoji = {'user': '👤', 'assistant': '🤖', 'system': '⚙️'}.get(msg['role'], '❓')
    text = msg['text']
    prefix = ""
    if '[Generator]' in text:
        prefix = "🤖 "
    elif '[Reviewer]' in text:
        prefix = "👁️  "
    elif '[SkillAgent]' in text:
        prefix = "🛠️  "
    display = text if len(text) <= 120 else text[:120] + "..."
    print(f"\n{i}. {role_emoji} {msg['role'].upper()}: {prefix}{display}")

print("\n" + "="*70)


🛠️  Skill実行ログ [Phase 3]

1. 🛠️  Skill: get_inventory
   ⏰ 時刻: 2026-03-20T03:32:36.147392
   📥 パラメータ: {'device_ip': '172.20.100.21', 'username': 'admin', 'password': 'admin@123', 'port': '830'}
   📤 結果: {'status': 'success', 'vlans': [{'name': 'MGMT_VLAN', 'vlan_id': '100'}], 'vlan_names': ['MGMT_VLAN'], 'raw_config': '\n## Last changed: 2026-03-20 03:30:29 UTC\nvlans {\n    MGMT_VLAN {\n        vlan

2. 🛠️  Skill: validate_xml
   ⏰ 時刻: 2026-03-20T03:32:37.147650
   📥 パラメータ: {'xml_config': '<configuration>\n    <vlans>\n        <vlan>\n       ...'}
   📤 結果: {'valid': True, 'errors': [], 'warnings': []}

3. 🛠️  Skill: deploy_netconf
   ⏰ 時刻: 2026-03-20T03:32:39.441588
   📥 パラメータ: {'xml_config': '<configuration>\n    <vlans>\n        <vlan>\n       ...', 'device_ip': '172.20.100.21', 'username': 'admin', 'password': 'admin@123', 'port': '830'}
   📤 結果: {'status': 'success', 'diff': '\n[edit vlans]\n+   SALES_VLAN {\n+       vlan-id 70;\n+   }\n', 'message': 'Configuration deployed succes

## 🔬 Skillの単体テスト

SkillはクラスMetohドと違い、独立した純粋関数なので単体テストが容易

In [11]:
print("\n" + "="*70)
print("🔬 Skill単体テスト")
print("="*70)

# テスト1: 正常な削除XML
print("\n--- テスト1: 正常な削除XML ---")
valid_delete_xml = """<configuration>
  <vlans>
    <vlan operation=\"delete\">
      <n>SALES_VLAN</n>
    </vlan>
  </vlans>
</configuration>"""
t1 = validate_xml_skill.execute(xml_config=valid_delete_xml)
print(f"valid: {t1['valid']}, errors: {t1['errors']}, warnings: {t1['warnings']}")
assert t1['valid'] == True, "テスト1失敗"
print("✅ テスト1 PASS")

# テスト2: <vlans>なし（構造エラー）
print("\n--- テスト2: <vlans>なし（構造エラー）---")
bad_xml = """<configuration>
  <vlan operation=\"delete\">
    <n>SALES_VLAN</n>
  </vlan>
</configuration>"""
t2 = validate_xml_skill.execute(xml_config=bad_xml)
print(f"valid: {t2['valid']}, errors: {t2['errors']}")
assert t2['valid'] == False, "テスト2失敗"
print("✅ テスト2 PASS")

# テスト3: fix_xml Skillで修正
print("\n--- テスト3: fix_xml Skillで構造修正 ---")
t3 = fix_xml_skill.execute(xml_config=bad_xml, translated_query="delete vlan")
print(f"changes: {t3['changes']}")
print(f"fixed_xml: {t3['fixed_xml'][:100]}...")

# 修正後に再検証
t3_revalidate = validate_xml_skill.execute(xml_config=t3['fixed_xml'])
print(f"再検証 valid: {t3_revalidate['valid']}")
assert t3_revalidate['valid'] == True, "テスト3失敗"
print("✅ テスト3 PASS")

# テスト4: vlan-idあり削除（警告）
print("\n--- テスト4: <vlan-id>あり削除（警告 + 自動修正）---")
dirty_xml = """<configuration>
  <vlans>
    <vlan operation=\"delete\">
      <n>SALES_VLAN</n>
      <vlan-id>70</vlan-id>
    </vlan>
  </vlans>
</configuration>"""
t4_val = validate_xml_skill.execute(xml_config=dirty_xml)
print(f"検証: valid={t4_val['valid']}, warnings={t4_val['warnings']}")
t4_fix = fix_xml_skill.execute(xml_config=dirty_xml, translated_query="delete vlan")
print(f"修正: changes={t4_fix['changes']}")
print("✅ テスト4 PASS")

print("\n" + "="*70)
print("🎉 全Skill単体テスト PASS")
print("="*70)


🔬 Skill単体テスト

--- テスト1: 正常な削除XML ---
valid: True, errors: [], warnings: []
✅ テスト1 PASS

--- テスト2: <vlans>なし（構造エラー）---
valid: False, errors: ['<vlan> must be under <vlans>: missing <vlans> parent tag']
✅ テスト2 PASS

--- テスト3: fix_xml Skillで構造修正 ---
changes: ['Fixed: Added missing <vlans> wrapper around <vlan>', "Fixed: Renamed <n>'SALES_VLAN'</n> to <name> (Junos NETCONF standard)"]
fixed_xml: <configuration>
  <vlans><vlan operation="delete">
    <name>SALES_VLAN</name></vlan>
</vlans></conf...
再検証 valid: True
✅ テスト3 PASS

--- テスト4: <vlan-id>あり削除（警告 + 自動修正）---
検証: valid=False, warnings=[]
修正: changes=["Fixed: Renamed <n>'SALES_VLAN'</n> to <name> (Junos NETCONF standard)", 'Fixed: Removed <vlan-id> from delete operation (only <name> allowed)']
✅ テスト4 PASS

🎉 全Skill単体テスト PASS


## 📈 Phase 2〜5 比較


In [12]:
print("""
╔══════════════════╦══════════════╦══════════════╦══════════════╦══════════════════════╗
║ 項目             ║ Phase 2      ║ Phase 3      ║ Phase 4      ║ Phase 5              ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ RAG検索          ║ 固定・毎回   ║ 固定・毎回   ║ 固定・毎回   ║ ✅ Skill化           ║
║                  ║              ║              ║              ║  能動的・必要時のみ  ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ デバイス状態     ║ 不明         ║ 不明         ║ 不明         ║ ✅ get_inventory     ║
║ 把握             ║              ║              ║              ║  事前にVLAN一覧取得  ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ 思考モデル       ║ 情報供給型   ║ 情報供給型   ║ 情報供給型   ║ ✅ ReAct型           ║
║                  ║              ║              ║              ║  Reason → Act        ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ XML検証          ║ -            ║ ✅ Skill      ║ ✅ 継承      ║ ✅ 継承              ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ ロールバック     ║ なし         ║ なし         ║ ✅ Skill      ║ ✅ 継承              ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ 監査             ║ なし         ║ なし         ║ ✅ Skill      ║ ✅ 継承              ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ Skill数          ║ -            ║ 3            ║ 5            ║ 7                    ║
╚══════════════════╩══════════════╩══════════════╩══════════════╩══════════════════════╝
""")

print("\n🔮 次のフェーズ (Phase 6) への道筋:")
print("   - normalize_vlan_name_skill: inventory結果でVLAN名の大文字/小文字を正規化")
print("   - diff_check_skill: commit前にshow | compareで差分を明示的に確認")
print("   - memory_skill: 過去の成功/失敗パターンをEmbeddingで記憶")



╔══════════════════╦══════════════╦══════════════╦══════════════╦══════════════════════╗
║ 項目             ║ Phase 2      ║ Phase 3      ║ Phase 4      ║ Phase 5              ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ RAG検索          ║ 固定・毎回   ║ 固定・毎回   ║ 固定・毎回   ║ ✅ Skill化           ║
║                  ║              ║              ║              ║  能動的・必要時のみ  ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ デバイス状態     ║ 不明         ║ 不明         ║ 不明         ║ ✅ get_inventory     ║
║ 把握             ║              ║              ║              ║  事前にVLAN一覧取得  ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ 思考モデル       ║ 情報供給型   ║ 情報供給型   ║ 情報供給型   ║ ✅ ReAct型           ║
║                  ║              ║              ║              ║  Reason → Act        ║
╠══════════════════╬══════════════╬══════════════╬══════════════╬══════════════════════╣
║ XML

## 🆕 Phase 6: Reviewer プロンプト再設計（Step 1）

設計書 §5・§6 Step1 に対応。Junos 仕様ルールを削除し「意図整合性チェック」に限定する。

> **影響範囲**: `NetconfRagWorkflowPhase3._initialize_agents()` 内の `reviewer_instructions` のみ


In [13]:
# ============================================================
# Phase 6 Step 1: Reviewer プロンプト再設計
# ============================================================
# 修正方針（設計書 §5）:
#   Before: Junos 仕様ルール（vlan-id の有無など）をプロンプトに含む
#           → Junos を知らない Reviewer が正しい XML を誤 REJECT するリスク
#   After:  「ユーザーの意図と生成 XML の操作種別・対象 VLAN が一致しているか」のみ判断
#           → Junos 仕様の判断は validate_xml Skill（コード）に完全移譲

REVIEWER_INSTRUCTIONS_V6 = """
You are an XML intent validator — NOT a Junos syntax expert.

[YOUR ONLY JOB]
Check whether the generated XML matches the user's stated intent.
Do NOT apply Junos-specific rules. Do NOT judge technical syntax.
Technical correctness is handled by the validate_xml skill (code), not by you.

[THREE CHECKS ONLY]
1. Operation type match:
   - User said "delete/remove" → XML must have operation="delete"
   - User said "add/create" → XML must NOT have operation="delete"
2. VLAN name match:
   - The VLAN name in XML must match the target VLAN from the user request
   - If inventory was provided, the VLAN name must exist in vlan_names (for delete)
3. Basic XML parsability:
   - XML must be well-formed (parseable), not a syntax error

[WHAT YOU MUST IGNORE]
- Whether <vlan-id> is present or absent
- Whether <vlans> wrapper exists
- Any other Junos-specific structural rules
- These are handled by validate_xml skill

[OUTPUT FORMAT - MANDATORY]
First word MUST be exactly one of: APPROVE, IMPROVE, REJECT

APPROVE: Intent matches — operation type and VLAN name are correct.
IMPROVE: <specific intent mismatch only>
REJECT: <critical intent mismatch — wrong operation type or wrong VLAN name>
"""

print("✅ REVIEWER_INSTRUCTIONS_V6 定義完了")
print("   Junos仕様ルール: 削除済み")
print("   判断基準: 意図整合性チェックのみ（操作種別・VLAN名・パース可能性）")


✅ REVIEWER_INSTRUCTIONS_V6 定義完了
   Junos仕様ルール: 削除済み
   判断基準: 意図整合性チェックのみ（操作種別・VLAN名・パース可能性）


## 🆕 Phase 6: task_decomposer Skill（Step 2）

設計書 §3.1・§6 Step2 に対応。  
自然言語入力を操作リスト JSON に分解する LLM ベースの Skill。

出力スキーマ:
```json
{
  "tasks": [
    { "id": "task_1", "operation": "delete", "target_vlan": "VLAN70", "depends_on": [] },
    { "id": "task_2", "operation": "create", "target_vlan": "VLAN100", "depends_on": [] },
    { "id": "task_3", "operation": "configure_interface", "target_vlan": "VLAN100",
      "interface": "xe-0/0/1", "depends_on": ["task_2"] }
  ]
}
```


In [14]:
# ============================================================
# Phase 6 Skill 8: task_decomposer
# 自然言語入力 → タスクリスト JSON（LLMベース）
# ============================================================

TASK_DECOMPOSER_PROMPT = """
You are a network operation task decomposer.
Analyze the user's network configuration request and break it down into atomic tasks.

[OUTPUT FORMAT]
Return ONLY valid JSON. No explanation. No markdown code blocks.
Schema:
{
  "tasks": [
    {
      "id": "task_1",
      "operation": "delete" | "create" | "configure_interface",
      "target_vlan": "<VLAN name or null>",
      "vlan_id": "<VLAN ID or null>",
      "interface": "<interface name or null>",
      "description": "<one-line description>",
      "depends_on": ["task_N", ...]
    }
  ]
}

[DEPENDENCY RULES]
- Interface configuration on a VLAN depends on that VLAN's create task.
- VLAN delete depends on any interface task that removes that VLAN first.
- Independent tasks have empty depends_on: [].
- Never create circular dependencies.

[CURRENT NETWORK STATE — USE THIS TO MAKE SMART DECISIONS]
{inventory_section}

[SMART DECISION RULES based on current state]
- If asked to DELETE a VLAN that does NOT exist in current state: set operation to "skip" and description to "Already absent — skip".
- If asked to CREATE a VLAN that ALREADY exists in current state: set operation to "skip" and description to "Already exists — skip".
- Only generate actionable tasks that actually need to change the device state.

[EXAMPLES]
Input: "VLAN70を削除して、VLAN100を追加する"
Output: {"tasks": [
  {"id":"task_1","operation":"delete","target_vlan":"VLAN70","vlan_id":null,"interface":null,"description":"Delete VLAN70","depends_on":[]},
  {"id":"task_2","operation":"create","target_vlan":"VLAN100","vlan_id":"100","interface":null,"description":"Create VLAN100","depends_on":[]}
]}

Input: "xe-0/0/1からVLAN70を外してからVLAN70を削除する"
Output: {"tasks": [
  {"id":"task_1","operation":"configure_interface","target_vlan":"VLAN70","vlan_id":null,"interface":"xe-0/0/1","description":"Remove VLAN70 from xe-0/0/1","depends_on":[]},
  {"id":"task_2","operation":"delete","target_vlan":"VLAN70","vlan_id":null,"interface":null,"description":"Delete VLAN70","depends_on":["task_1"]}
]}
"""

def decompose_tasks(
    user_query: str,
    llm=None,
    inventory: Dict = None,   # ★ v4: Orchestrator から最新 Inventory を受け取る
) -> Dict[str, Any]:
    """
    自然言語入力をタスクリスト JSON に分解する（Phase 6 新規）

    LLM に操作リスト JSON を返すよう指示するプロンプトを使用。
    出力は { tasks: [ { id, operation, target_vlan, depends_on } ] } の形式。

    Returns:
        {
            "status": "success" | "failure",
            "tasks": [ { id, operation, target_vlan, vlan_id, interface, description, depends_on } ],
            "raw_response": str,
            "message": str
        }
    """
    if llm is None:
        return {"status": "failure", "tasks": [], "raw_response": "",
                "message": "LLM not provided to task_decomposer"}
    try:
        # ★ v4: inventory セクションを動的に構築してプロンプトに注入
        if inventory and inventory.get("status") == "success":
            vlan_names = inventory.get("vlan_names", [])
            raw_cfg = inventory.get("raw_config", "").strip()
            inventory_section = (
                f"Existing VLANs on device: {vlan_names if vlan_names else '(none)'}\n"
                f"Raw config:\n{raw_cfg if raw_cfg else '(empty)'}"
            )
        else:
            inventory_section = "(Inventory not available — proceed based on user request only)"

        prompt_with_inventory = TASK_DECOMPOSER_PROMPT.replace(
            "{inventory_section}", inventory_section
        )
        prompt = f"{prompt_with_inventory}\n\nInput: {user_query}\nOutput:"
        response = llm.invoke(prompt)
        raw = response.content.strip()

        # JSON 抽出（```json ブロックがある場合も対応）
        json_match = re.search(r"```json\s*(.+?)\s*```", raw, re.DOTALL)
        if json_match:
            raw = json_match.group(1).strip()

        parsed = json.loads(raw)
        tasks = parsed.get("tasks", [])

        return {
            "status": "success",
            "tasks": tasks,
            "raw_response": raw,
            "message": f"Decomposed into {len(tasks)} task(s)"
        }
    except json.JSONDecodeError as e:
        return {"status": "failure", "tasks": [], "raw_response": raw if 'raw' in dir() else "",
                "message": f"JSON parse error: {e}"}
    except Exception as e:
        return {"status": "failure", "tasks": [], "raw_response": "",
                "message": f"task_decomposer error: {e}"}


task_decomposer_skill = Skill(
    name="task_decomposer",
    description=(
        "Decompose a natural language network configuration request into an ordered task list (JSON). "
        "Returns tasks(list), each with id/operation/target_vlan/depends_on. "
        "Use as the first step in Orchestrator before dependency_resolver."
    ),
    function=decompose_tasks,
    parameters={
        "user_query": {"type": "string", "description": "Natural language user request (Japanese or English)"},
        "llm": {"type": "object", "description": "ChatOpenAI instance for LLM inference"},
        "inventory": {"type": "dict", "description": "Result of get_inventory Skill (optional, for smart skip decisions)"}
    }
)

print("✅ task_decomposer Skill 定義完了 (v4: inventory注入対応)")
print("   入力: 自然言語クエリ + inventory（現在のVLAN状態）")
print("   出力: { tasks: [ { id, operation, target_vlan, vlan_id, interface, depends_on } ] }")
print("   スマート判定: 既存VLANへのcreate / 不在VLANへのdelete → skip")


✅ task_decomposer Skill 定義完了 (v4: inventory注入対応)
   入力: 自然言語クエリ + inventory（現在のVLAN状態）
   出力: { tasks: [ { id, operation, target_vlan, vlan_id, interface, depends_on } ] }
   スマート判定: 既存VLANへのcreate / 不在VLANへのdelete → skip


## 🆕 Phase 6: dependency_resolver Skill（Step 3）

設計書 §3.2・§6 Step3 に対応。  
タスクリストの依存関係をトポロジカルソートし、実行可能な順序を返す。

- 循環依存を検出した場合はエラーとして報告
- 依存のないタスクは `parallel: true` とマーク（将来的な並列化の基盤）


In [15]:
# ============================================================
# Phase 6 Skill 9: dependency_resolver
# タスクリストをトポロジカルソートして実行順序を決定
# ============================================================

def resolve_dependencies(tasks: List[Dict]) -> Dict[str, Any]:
    """
    タスクリストの依存関係を解析し、実行順序を決定する（Phase 6 新規）

    トポロジカルソート（Kahn's algorithm）で DAG を解決。
    循環依存を検出した場合はエラーを返す。
    依存のないタスクには parallel=True をマーク（将来の並列化基盤）。

    Args:
        tasks: task_decomposer が返した tasks リスト

    Returns:
        {
            "status": "success" | "error",
            "execution_order": [ task_dict, ... ],  # 実行順（各タスクに parallel フラグ付き）
            "message": str
        }
    """
    if not tasks:
        return {"status": "success", "execution_order": [], "message": "No tasks to resolve"}

    # ID → タスク のマップを作成
    task_map = {t["id"]: t for t in tasks}
    task_ids = set(task_map.keys())

    # 存在しないタスク ID への依存をチェック
    for t in tasks:
        for dep in t.get("depends_on", []):
            if dep not in task_ids:
                return {
                    "status": "error",
                    "execution_order": [],
                    "message": f"Unknown dependency: task '{t['id']}' depends on '{dep}' which does not exist"
                }

    # Kahn's algorithm でトポロジカルソート
    from collections import deque

    in_degree = {t["id"]: 0 for t in tasks}
    dependents = {t["id"]: [] for t in tasks}  # dep → [tasks that depend on dep]

    for t in tasks:
        for dep in t.get("depends_on", []):
            in_degree[t["id"]] += 1
            dependents[dep].append(t["id"])

    queue = deque([tid for tid, deg in in_degree.items() if deg == 0])
    execution_order = []

    while queue:
        tid = queue.popleft()
        task = copy.deepcopy(task_map[tid])
        task["parallel"] = (len(task.get("depends_on", [])) == 0 and
                            sum(1 for t in tasks if len(t.get("depends_on", [])) == 0) > 1)
        execution_order.append(task)
        for dependent_id in dependents[tid]:
            in_degree[dependent_id] -= 1
            if in_degree[dependent_id] == 0:
                queue.append(dependent_id)

    # 循環依存チェック
    if len(execution_order) != len(tasks):
        remaining = [tid for tid, deg in in_degree.items() if deg > 0]
        return {
            "status": "error",
            "execution_order": [],
            "message": f"Circular dependency detected among tasks: {remaining}"
        }

    return {
        "status": "success",
        "execution_order": execution_order,
        "message": f"Resolved {len(execution_order)} task(s) in execution order"
    }


dependency_resolver_skill = Skill(
    name="dependency_resolver",
    description=(
        "Resolve task dependencies using topological sort (Kahn's algorithm). "
        "Detects circular dependencies and returns ordered execution plan. "
        "Marks independent tasks with parallel=True for future parallel execution. "
        "Returns execution_order(list), status, message."
    ),
    function=resolve_dependencies,
    parameters={
        "tasks": {"type": "list", "description": "Task list from task_decomposer (with id and depends_on)"}
    }
)

print("✅ dependency_resolver Skill 定義完了")
print("   アルゴリズム: Kahn's algorithm（トポロジカルソート）")
print("   循環依存検出: ✅")
print("   parallel フラグ: ✅（将来の並列化基盤）")


✅ dependency_resolver Skill 定義完了
   アルゴリズム: Kahn's algorithm（トポロジカルソート）
   循環依存検出: ✅
   parallel フラグ: ✅（将来の並列化基盤）


## 🆕 Phase 6: result_aggregator Skill（Step 5）

設計書 §3.3・§6 Step5 に対応。  
全 Task の実行結果を集約し、ユーザー向けの最終レポートを生成する。

- 各 Task の deploy diff・audit エビデンスをまとめて出力
- 失敗 Task がある場合、どのタスクが失敗したか・影響範囲を明示


In [16]:
# ============================================================
# Phase 6 Skill 10: result_aggregator
# 全 Task の実行結果を集約・最終レポート生成
# ============================================================

def aggregate_results(task_results: List[Dict]) -> Dict[str, Any]:
    """
    全 Task の実行結果を集約してユーザー向けレポートを生成する（Phase 6 新規）

    Args:
        task_results: 各タスクの実行結果リスト
            [ { "task": task_dict, "result": worker_result_dict }, ... ]

    Returns:
        {
            "status": "all_success" | "partial_failure" | "all_failure",
            "summary": str,           # ユーザー向け要約テキスト
            "succeeded_tasks": list,
            "failed_tasks": list,
            "skipped_tasks": list,
            "report_lines": [str],    # 詳細レポート行
        }
    """
    succeeded, failed, skipped = [], [], []
    report_lines = []

    report_lines.append("=" * 60)
    report_lines.append("📊 Phase 6 Orchestrator 実行レポート")
    report_lines.append("=" * 60)

    for entry in task_results:
        task = entry.get("task", {})
        result = entry.get("result", {})
        tid = task.get("id", "unknown")
        op = task.get("operation", "?")
        vlan = task.get("target_vlan", "?")
        iface = task.get("interface", "")
        desc = task.get("description", "")

        deploy = result.get("deployment_status", {}) or {}
        audit = result.get("audit_status") or {}
        rollback = result.get("rollback_status") or {}
        valid = result.get("validation_status", False)

        deploy_status = deploy.get("status", "unknown")
        audit_status = audit.get("status", "") if audit else ""

        task_label = f"[{tid}] {op} / {vlan}" + (f" / {iface}" if iface else "")
        report_lines.append(f"\n  📋 {task_label}")
        report_lines.append(f"     説明: {desc}")
        report_lines.append(f"     検証: {'✅' if valid else '❌'}")
        report_lines.append(f"     デプロイ: {deploy_status}")

        if deploy.get("diff"):
            for diff_line in deploy["diff"].strip().split("\n")[:5]:
                report_lines.append(f"     diff: {diff_line}")

        if audit_status:
            report_lines.append(f"     Audit: {audit_status} - {audit.get('message','')}")

        if rollback:
            report_lines.append(f"     Rollback: {rollback.get('status','')} [{rollback.get('mode','')}]")

        # 成功/失敗/スキップの分類
        if deploy_status in ("success", "no_changes") and (not audit_status or audit_status == "success"):
            succeeded.append(tid)
            report_lines.append(f"     → ✅ SUCCESS")
        elif deploy_status == "skipped":
            skipped.append(tid)
            report_lines.append(f"     → ⏭️  SKIPPED")
        else:
            failed.append(tid)
            report_lines.append(f"     → ❌ FAILED")

    report_lines.append("\n" + "=" * 60)

    # サマリー決定
    if not failed and not skipped:
        overall = "all_success"
        summary = f"✅ 全 {len(succeeded)} タスク成功"
    elif not failed and not succeeded and skipped:
        # 全タスクがスキップ = deploy=False のドライラン
        overall = "dry_run_complete"
        summary = f"🔍 ドライラン完了: 全 {len(skipped)} タスクの XML 生成・検証が成功 (デプロイ未実行)"
    elif failed and not succeeded:
        overall = "all_failure"
        summary = f"❌ 全 {len(failed)} タスク失敗"
    else:
        overall = "partial_failure"
        summary = (f"⚠️ {len(succeeded)} 成功 / {len(failed)} 失敗 / {len(skipped)} スキップ")
        if failed:
            summary += f"\n   失敗タスク: {failed}"

    report_lines.append(f"  {summary}")
    report_lines.append("=" * 60)

    return {
        "status": overall,
        "summary": summary,
        "succeeded_tasks": succeeded,
        "failed_tasks": failed,
        "skipped_tasks": skipped,
        "report_lines": report_lines
    }


result_aggregator_skill = Skill(
    name="result_aggregator",
    description=(
        "Aggregate results from all Worker tasks and generate final user report. "
        "Summarizes deploy diff, audit evidence, rollback status per task. "
        "Returns status(all_success/partial_failure/all_failure), summary, report_lines."
    ),
    function=aggregate_results,
    parameters={
        "task_results": {
            "type": "list",
            "description": "List of {task: task_dict, result: worker_result_dict}"
        }
    }
)

print("✅ result_aggregator Skill 定義完了")
print("   出力: all_success / partial_failure / all_failure")
print("   レポート: deploy diff + audit エビデンス + rollback 状況")


✅ result_aggregator Skill 定義完了
   出力: all_success / partial_failure / all_failure
   レポート: deploy diff + audit エビデンス + rollback 状況


## 🆕 Phase 6: Skill 登録更新（累計 10 個）


In [17]:
import json
import copy

# Phase 6 追加 Skills を ALL_SKILLS に追加
PHASE6_SKILLS = [
    task_decomposer_skill,
    dependency_resolver_skill,
    result_aggregator_skill
]

ALL_SKILLS_V6 = ALL_SKILLS + PHASE6_SKILLS

print("✅ Phase 6 Skill登録完了")
print(f"   Phase 5 Skills: {len(ALL_SKILLS)}個")
print(f"   Phase 6 追加 : {len(PHASE6_SKILLS)}個")
print(f"   累計          : {len(ALL_SKILLS_V6)}個")
print()
for sk in ALL_SKILLS_V6:
    tag = " ★ NEW" if sk in PHASE6_SKILLS else ""
    print(f"   🛠️  {sk.name}{tag}")


✅ Phase 6 Skill登録完了
   Phase 5 Skills: 7個
   Phase 6 追加 : 3個
   累計          : 10個

   🛠️  validate_xml
   🛠️  fix_xml
   🛠️  deploy_netconf
   🛠️  rollback
   🛠️  audit
   🛠️  get_inventory
   🛠️  lookup_documentation
   🛠️  task_decomposer ★ NEW
   🛠️  dependency_resolver ★ NEW
   🛠️  result_aggregator ★ NEW


## 🆕 Phase 6: OrchestratorAgent クラス（Step 4）

設計書 §2・§6 Step4 に対応。  
`task_decomposer → dependency_resolver → Worker 順次ディスパッチ → result_aggregator` を統合する。

**Worker は `NetconfRagWorkflowPhase3` をそのまま使用（変更なし）。**


In [18]:
# ============================================================
# Phase 6: OrchestratorAgent
# Orchestrator-Worker パターンの統合クラス
# ============================================================

class OrchestratorAgent:
    """
    NETCONF RAG Orchestrator（Phase 6 新規）

    責務:
      1. task_decomposer で自然言語 → タスクリスト JSON に分解
      2. dependency_resolver で依存関係を解決・実行順序を決定
      3. 各 Task を NetconfRagWorkflowPhase3 (Worker) へ順次ディスパッチ
      4. result_aggregator で全 Task 結果を集約・最終レポートを生成

    Worker (Phase 5) の内部実装は一切変更しない。
    タスク数が 1 の場合も同じフローで処理する（Phase 5 との互換性維持）。
    """

    def __init__(
        self,
        retriever,
        llm,
        skills: List[Skill] = None,
        max_retries: int = 3,
        max_review_rounds: int = 2,
    ):
        self.retriever = retriever
        self.llm = llm
        self.max_retries = max_retries
        self.max_review_rounds = max_review_rounds
        self.logs: List[str] = []

        # Skill 辞書（Phase 6 全 10 Skills）
        self.skills: Dict[str, Skill] = {}
        for sk in (skills or ALL_SKILLS_V6):
            self.skills[sk.name] = sk

        print("✅ OrchestratorAgent 初期化完了")
        print(f"   登録 Skills: {list(self.skills.keys())}")

    def log(self, message: str):
        self.logs.append(message)
        print(message)

    def _run_skill(self, skill_name: str, **kwargs):
        if skill_name not in self.skills:
            self.log(f"  ❌ [Skill] 未知: {skill_name}")
            return None
        skill = self.skills[skill_name]
        self.log(f"  🛠️  [Skill:{skill_name}] 実行中...")
        try:
            result = skill.execute(**kwargs)
            self.log(f"  ✅ [Skill:{skill_name}] 完了")
            return result
        except Exception as e:
            self.log(f"  ❌ [Skill:{skill_name}] エラー: {e}")
            return None

    def _build_worker_query(self, task: Dict) -> str:
        """
        タスク dict からWorker に渡す自然言語クエリを再構築する。
        task_decomposer が分解した情報を使い、Worker が1操作として処理できる形に整形。
        """
        op = task.get("operation", "")
        vlan = task.get("target_vlan", "")
        vlan_id = task.get("vlan_id", "")
        iface = task.get("interface", "")
        description = task.get("description", "")

        # description が充実していればそのまま使用（英語）
        if description:
            if op == "delete":
                return f"Delete VLAN named {vlan}."
            elif op == "create":
                vid_part = f" with VLAN ID {vlan_id}" if vlan_id else ""
                return f"Create VLAN named {vlan}{vid_part}."
            elif op == "configure_interface":
                return f"Configure interface {iface} to use VLAN {vlan}."
            else:
                return description

        return f"{op} {vlan}" + (f" on {iface}" if iface else "")

    async def run(
        self,
        user_query: str,
        device_ip: str = None,
        username: str = None,
        password: str = None,
        port: str = "830",
        deploy: bool = False,
    ) -> Dict[str, Any]:
        """
        Orchestrator メインフロー

        Returns:
            {
                "user_query": str,
                "tasks": list,           # decomposer が生成したタスクリスト
                "execution_order": list, # resolver が決定した実行順
                "task_results": list,    # 各 Worker の結果
                "aggregated": dict,      # result_aggregator の出力
                "orchestrator_logs": list
            }
        """
        self.logs = []

        self.log("\n" + "=" * 70)
        self.log("🎼 Phase 6: Orchestrator 起動")
        self.log("=" * 70)
        self.log(f"入力クエリ: {user_query}")

        result = {
            "user_query": user_query,
            "tasks": [],
            "execution_order": [],
            "task_results": [],
            "aggregated": {},
            "orchestrator_logs": self.logs
        }

        # ── Step 0: get_inventory（task_decomposer 用の最新状態取得）──
        # ★ v4: Orchestrator レベルでも実機の現在状態を先に取得する。
        #        task_decomposer が skip 判断など賓い分解を行えるようにする。
        orchestrator_inventory = None
        if all([device_ip, username, password]):
            self.log("\n" + "─" * 50)
            self.log("🗂️  [Step 0] get_inventory: Orchestrator用 現在状態取得")
            inv = self._run_skill(
                "get_inventory",
                device_ip=device_ip,
                username=username,
                password=password,
                port=port
            )
            if inv and inv.get("status") == "success":
                orchestrator_inventory = inv
                vlan_names = inv.get("vlan_names", [])
                self.log(f"   現在のVLAN一覧: {vlan_names if vlan_names else '(空)'}")
                result["orchestrator_inventory"] = inv
            else:
                msg = inv.get("message", "") if inv else "Skill error"
                self.log(f"   ⚠️  Inventory取得失敗（続行）: {msg}")

        # ── Step 1: task_decomposer ──────────────────────────────
        self.log("\n" + "─" * 50)
        self.log("📋 [Step 1] task_decomposer: タスク分解（現在状態を踏まえて）")
        decompose_result = self._run_skill(
            "task_decomposer",
            user_query=user_query,
            llm=self.llm,
            inventory=orchestrator_inventory   # ★ v4: 最新 Inventory を注入
        )

        if not decompose_result or decompose_result["status"] != "success":
            msg = decompose_result.get("message", "unknown") if decompose_result else "Skill error"
            self.log(f"❌ task_decomposer 失敗: {msg}")
            result["aggregated"] = {"status": "all_failure", "summary": f"task_decomposer 失敗: {msg}",
                                    "report_lines": []}
            return result

        tasks = decompose_result["tasks"]
        result["tasks"] = tasks
        self.log(f"   → {len(tasks)} タスク検出")
        for t in tasks:
            dep_str = f" (依存: {t.get('depends_on', [])})" if t.get("depends_on") else ""
            self.log(f"      {t['id']}: {t.get('operation','?')} / {t.get('target_vlan','?')} {dep_str}")

        # ── Step 2: dependency_resolver ─────────────────────────
        self.log("\n" + "─" * 50)
        self.log("🔗 [Step 2] dependency_resolver: 依存関係解決")
        resolve_result = self._run_skill("dependency_resolver", tasks=tasks)

        if not resolve_result or resolve_result["status"] != "success":
            msg = resolve_result.get("message", "unknown") if resolve_result else "Skill error"
            self.log(f"❌ dependency_resolver 失敗: {msg}")
            result["aggregated"] = {"status": "all_failure",
                                    "summary": f"dependency_resolver 失敗: {msg}", "report_lines": []}
            return result

        execution_order = resolve_result["execution_order"]
        result["execution_order"] = execution_order
        self.log(f"   → 実行順: {[t['id'] for t in execution_order]}")
        parallel_tasks = [t["id"] for t in execution_order if t.get("parallel")]
        if parallel_tasks:
            self.log(f"   → 並列可能タスク（将来対応）: {parallel_tasks}")

        # ── Step 3: Worker への順次ディスパッチ ─────────────────
        self.log("\n" + "─" * 50)
        self.log(f"🚀 [Step 3] Worker ディスパッチ開始（{len(execution_order)} タスク）")
        task_results = []

        for idx, task in enumerate(execution_order, 1):
            tid = task["id"]
            self.log(f"\n  [{idx}/{len(execution_order)}] 🔧 Worker 起動: {tid}")
            self.log(f"      operation: {task.get('operation','?')} | vlan: {task.get('target_vlan','?')} | interface: {task.get('interface','-')}")

            # ★ v4: operation=skip はWorkerにディスパッチせず即完了扱い
            if task.get("operation") == "skip":
                self.log(f"  ⏭️  [{idx}/{len(execution_order)}] SKIP: {tid} — {task.get('description', '')}")
                task_results.append({
                    "task": task,
                    "result": {
                        "validation_status": True,
                        "deployment_status": {
                            "status": "no_changes",
                            "diff": "",
                            "message": f"Skipped by task_decomposer: {task.get('description', '')}"
                        },
                        "audit_status": None,
                        "rollback_status": None
                    }
                })
                continue

            # Worker 用クエリ生成
            worker_query = self._build_worker_query(task)
            self.log(f"      Worker クエリ: {worker_query}")

            # Phase 5 Worker インスタンス生成（設計書: Worker は変更なし）
            worker = NetconfRagWorkflowPhase3(
                retriever=self.retriever,
                llm=self.llm,
                skills=ALL_SKILLS,           # Phase 5 Skills のみ渡す（Phase 6 Skills は Orchestrator が持つ）
                max_retries=self.max_retries,
                max_review_rounds=self.max_review_rounds,
            )

            # Reviewer プロンプトを Phase 6 版に差し替え（設計書 §5）
            worker.xml_reviewer.instructions = REVIEWER_INSTRUCTIONS_V6

            # Worker 実行
            try:
                worker_result = await worker.run(
                    user_query=worker_query,
                    device_ip=device_ip,
                    username=username,
                    password=password,
                    port=port,
                    deploy=deploy
                )
            except Exception as e:
                self.log(f"  ❌ Worker エラー: {e}")
                worker_result = {
                    "validation_status": False,
                    "deployment_status": {"status": "failure", "diff": "", "message": str(e)},
                    "audit_status": None, "rollback_status": None
                }

            deploy_status = worker_result.get("deployment_status", {})
            deploy_ok = deploy_status.get("status") in ("success", "no_changes", "skipped")
            audit_ok = (worker_result.get("audit_status") or {}).get("status", "success") in ("success", "skipped", "")

            task_success = deploy_ok and audit_ok
            self.log(f"  {'✅' if task_success else '❌'} Worker 完了: {tid} → deploy={deploy_status.get('status','?')} audit={( worker_result.get('audit_status') or {}).get('status','-')}")

            task_results.append({"task": task, "result": worker_result})

            # 失敗した場合は残タスクを中断（設計書 §2.1）
            if not task_success:
                self.log(f"\n  🛑 タスク {tid} 失敗 → 残タスクを中断")
                remaining = execution_order[idx:]  # 未実行タスク
                for rem_task in remaining:
                    task_results.append({
                        "task": rem_task,
                        "result": {
                            "validation_status": False,
                            "deployment_status": {
                                "status": "skipped",
                                "diff": "",
                                "message": f"Skipped due to failure of {tid}"
                            },
                            "audit_status": None, "rollback_status": None
                        }
                    })
                break

        result["task_results"] = task_results

        # ── Step 4: result_aggregator ────────────────────────────
        self.log("\n" + "─" * 50)
        self.log("📊 [Step 4] result_aggregator: 結果集約")
        aggregated = self._run_skill("result_aggregator", task_results=task_results)
        result["aggregated"] = aggregated or {}

        if aggregated:
            self.log(f"\n{aggregated['summary']}")
            for line in aggregated.get("report_lines", []):
                self.log(line)

        self.log("\n" + "=" * 70)
        self.log("🎼 Orchestrator 完了")
        self.log("=" * 70)

        return result


print("✅ OrchestratorAgent クラス定義完了")
print("   Phase 5 Worker: 変更なし（NetconfRagWorkflowPhase3 をそのまま使用）")
print("   Reviewer プロンプト: Worker 生成時に Phase 6 版（意図整合性のみ）に差し替え")


✅ OrchestratorAgent クラス定義完了
   Phase 5 Worker: 変更なし（NetconfRagWorkflowPhase3 をそのまま使用）
   Reviewer プロンプト: Worker 生成時に Phase 6 版（意図整合性のみ）に差し替え


## ⚙️ Phase 6 設定


In [19]:
# デバイス設定（Phase 5 継承）
DEVICE_CONFIG = {
    'ip': '192.0.2.1'  # Replace with your device IP,
    'port': '830',
    'username': 'admin',
    'password': 'YOUR_PASSWORD_HERE'
}

# ────────────────────────────────────────────
# テストクエリ（設計書 §4 の例に準拠）
# ────────────────────────────────────────────

# 例1: VLAN 削除 + 別 VLAN 追加（独立・依存なし）
USER_QUERY_P6 = "SALES_VLANのVLAN ID 70を削除して、DEV_VLANのVLAN ID 200を追加してください。"

# 例2: インターフェースから外してから削除（依存あり）
# USER_QUERY_P6 = "インターフェース xe-0/0/1 から SALES_VLAN を外してから、SALES_VLAN を削除してください。"

# 例3: 複合操作（設計書 §4 例3）
# USER_QUERY_P6 = "SALES_VLAN(ID70)を削除し、MGMT_VLAN(ID100)を追加して、xe-0/0/1にMGMT_VLANを設定してください。"

DEPLOY_TO_DEVICE_P6 = True    # ★ v3: deploy_netconf の "statement not found" を no_changes に修正済み
MAX_RETRIES_P6 = 3
MAX_REVIEW_ROUNDS_P6 = 2

print("✅ Phase 6 設定完了")
print(f"   クエリ  : {USER_QUERY_P6}")
print(f"   デプロイ: {DEPLOY_TO_DEVICE_P6}")


✅ Phase 6 設定完了
   クエリ  : SALES_VLANのVLAN ID 70を削除して、DEV_VLANのVLAN ID 200を追加してください。
   デプロイ: True


## 🚀 Phase 6 Orchestrator 実行


In [20]:
# Phase 6 Orchestrator 実行
orchestrator = OrchestratorAgent(
    retriever=retriever,
    llm=llm,
    skills=ALL_SKILLS_V6,
    max_retries=MAX_RETRIES_P6,
    max_review_rounds=MAX_REVIEW_ROUNDS_P6,
)

orch_result = await orchestrator.run(
    user_query=USER_QUERY_P6,
    device_ip=DEVICE_CONFIG['ip'],
    username=DEVICE_CONFIG['username'],
    password=DEVICE_CONFIG['password'],
    port=DEVICE_CONFIG['port'],
    deploy=DEPLOY_TO_DEVICE_P6
)

print("\n✅ Orchestrator 実行完了")


✅ OrchestratorAgent 初期化完了
   登録 Skills: ['validate_xml', 'fix_xml', 'deploy_netconf', 'rollback', 'audit', 'get_inventory', 'lookup_documentation', 'task_decomposer', 'dependency_resolver', 'result_aggregator']

🎼 Phase 6: Orchestrator 起動
入力クエリ: SALES_VLANのVLAN ID 70を削除して、DEV_VLANのVLAN ID 200を追加してください。

──────────────────────────────────────────────────
🗂️  [Step 0] get_inventory: Orchestrator用 現在状態取得
  🛠️  [Skill:get_inventory] 実行中...
  ✅ [Skill:get_inventory] 完了
   現在のVLAN一覧: ['MGMT_VLAN', 'SALES_VLAN']

──────────────────────────────────────────────────
📋 [Step 1] task_decomposer: タスク分解（現在状態を踏まえて）
  🛠️  [Skill:task_decomposer] 実行中...
  ✅ [Skill:task_decomposer] 完了
   → 2 タスク検出
      task_1: delete / SALES_VLAN 
      task_2: create / DEV_VLAN 

──────────────────────────────────────────────────
🔗 [Step 2] dependency_resolver: 依存関係解決
  🛠️  [Skill:dependency_resolver] 実行中...
  ✅ [Skill:dependency_resolver] 完了
   → 実行順: ['task_1', 'task_2']
   → 並列可能タスク（将来対応）: ['task_1', 'task_2']

───

## 📊 Phase 6 実行結果


In [21]:
print("\n" + "=" * 70)
print("📊 Phase 6 Orchestrator 実行サマリー")
print("=" * 70)

agg = orch_result.get("aggregated", {})
print(f"\n{agg.get('summary', '(サマリーなし)')}")
print(f"\n  検出タスク数  : {len(orch_result.get('tasks', []))}")
print(f"  実行順タスク数: {len(orch_result.get('execution_order', []))}")
print(f"  完了タスク数  : {len(orch_result.get('task_results', []))}")

print("\n" + "─" * 50)
print("📋 タスク分解結果 (task_decomposer)")
print("─" * 50)
for t in orch_result.get("tasks", []):
    dep = f" → depends on {t['depends_on']}" if t.get("depends_on") else ""
    print(f"  {t['id']}: [{t.get('operation','?')}] {t.get('target_vlan','?')} {dep}")

print("\n" + "─" * 50)
print("🔗 実行順 (dependency_resolver)")
print("─" * 50)
for i, t in enumerate(orch_result.get("execution_order", []), 1):
    par = " [並列可]" if t.get("parallel") else ""
    print(f"  {i}. {t['id']}: {t.get('operation','?')} / {t.get('target_vlan','?')} {par}")

print("\n" + "─" * 50)
print("📄 詳細レポート (result_aggregator)")
print("─" * 50)
for line in agg.get("report_lines", []):
    print(line)

print("\n" + "=" * 70)



📊 Phase 6 Orchestrator 実行サマリー

✅ 全 2 タスク成功

  検出タスク数  : 2
  実行順タスク数: 2
  完了タスク数  : 2

──────────────────────────────────────────────────
📋 タスク分解結果 (task_decomposer)
──────────────────────────────────────────────────
  task_1: [delete] SALES_VLAN 
  task_2: [create] DEV_VLAN 

──────────────────────────────────────────────────
🔗 実行順 (dependency_resolver)
──────────────────────────────────────────────────
  1. task_1: delete / SALES_VLAN  [並列可]
  2. task_2: create / DEV_VLAN  [並列可]

──────────────────────────────────────────────────
📄 詳細レポート (result_aggregator)
──────────────────────────────────────────────────
📊 Phase 6 Orchestrator 実行レポート

  📋 [task_1] delete / SALES_VLAN
     説明: Delete SALES_VLAN
     検証: ✅
     デプロイ: success
     diff: [edit vlans]
     diff: -   SALES_VLAN {
     diff: -       vlan-id 70;
     diff: -   }
     Audit: success - ✅ Confirmed: SALES_VLAN deleted from configuration
     → ✅ SUCCESS

  📋 [task_2] create / DEV_VLAN
     説明: Create DEV_VLAN
     検証: ✅
   

## 🔬 Phase 6 Skills 単体テスト


In [22]:
print("\n" + "=" * 70)
print("🔬 Phase 6 Skill 単体テスト")
print("=" * 70)

# ── テスト 1: dependency_resolver（独立タスク）──
print("\n--- テスト1: dependency_resolver（独立タスク）---")
tasks_independent = [
    {"id": "task_1", "operation": "delete", "target_vlan": "VLAN70", "depends_on": []},
    {"id": "task_2", "operation": "create", "target_vlan": "VLAN100", "depends_on": []},
]
r1 = resolve_dependencies(tasks_independent)
print(f"status: {r1['status']}")
print(f"order : {[t['id'] for t in r1['execution_order']]}")
print(f"parallel task_1: {r1['execution_order'][0].get('parallel')}")
assert r1["status"] == "success"
assert len(r1["execution_order"]) == 2
print("✅ テスト1 PASS")

# ── テスト 2: dependency_resolver（依存あり）──
print("\n--- テスト2: dependency_resolver（依存あり）---")
tasks_dependent = [
    {"id": "task_1", "operation": "configure_interface", "target_vlan": "VLAN70",
     "interface": "xe-0/0/1", "depends_on": []},
    {"id": "task_2", "operation": "delete", "target_vlan": "VLAN70", "depends_on": ["task_1"]},
]
r2 = resolve_dependencies(tasks_dependent)
print(f"status: {r2['status']}")
print(f"order : {[t['id'] for t in r2['execution_order']]}")
assert r2["status"] == "success"
assert r2["execution_order"][0]["id"] == "task_1"
assert r2["execution_order"][1]["id"] == "task_2"
print("✅ テスト2 PASS")

# ── テスト 3: dependency_resolver（循環依存検出）──
print("\n--- テスト3: dependency_resolver（循環依存）---")
tasks_circular = [
    {"id": "task_1", "operation": "delete", "target_vlan": "VLAN70", "depends_on": ["task_2"]},
    {"id": "task_2", "operation": "create", "target_vlan": "VLAN100", "depends_on": ["task_1"]},
]
r3 = resolve_dependencies(tasks_circular)
print(f"status: {r3['status']}")
print(f"message: {r3['message']}")
assert r3["status"] == "error"
assert "Circular" in r3["message"]
print("✅ テスト3 PASS（循環依存を正しく検出）")

# ── テスト 4: result_aggregator──
print("\n--- テスト4: result_aggregator ---")
mock_results = [
    {
        "task": {"id": "task_1", "operation": "delete", "target_vlan": "VLAN70",
                 "interface": None, "description": "Delete VLAN70", "depends_on": []},
        "result": {
            "validation_status": True,
            "deployment_status": {"status": "success", "diff": "[edit vlans]\n- VLAN70;", "message": "OK"},
            "audit_status": {"status": "success", "message": "Confirmed deleted", "vlan_name": "VLAN70",
                             "operation": "delete", "evidence": ""},
            "rollback_status": None
        }
    },
    {
        "task": {"id": "task_2", "operation": "create", "target_vlan": "VLAN100",
                 "interface": None, "description": "Create VLAN100", "depends_on": []},
        "result": {
            "validation_status": True,
            "deployment_status": {"status": "success", "diff": "[edit vlans]\n+ VLAN100 { vlan-id 100; }", "message": "OK"},
            "audit_status": {"status": "success", "message": "Confirmed added", "vlan_name": "VLAN100",
                             "operation": "create", "evidence": ""},
            "rollback_status": None
        }
    }
]
r4 = aggregate_results(mock_results)
print(f"status : {r4['status']}")
print(f"summary: {r4['summary']}")
print(f"succeeded: {r4['succeeded_tasks']}")
assert r4["status"] == "all_success"
assert len(r4["succeeded_tasks"]) == 2
print("✅ テスト4 PASS")

print("\n" + "=" * 70)
print("🎉 Phase 6 Skill 単体テスト 全 PASS")
print("=" * 70)



🔬 Phase 6 Skill 単体テスト

--- テスト1: dependency_resolver（独立タスク）---
status: success
order : ['task_1', 'task_2']
parallel task_1: True
✅ テスト1 PASS

--- テスト2: dependency_resolver（依存あり）---
status: success
order : ['task_1', 'task_2']
✅ テスト2 PASS

--- テスト3: dependency_resolver（循環依存）---
status: error
message: Circular dependency detected among tasks: ['task_1', 'task_2']
✅ テスト3 PASS（循環依存を正しく検出）

--- テスト4: result_aggregator ---
status : all_success
summary: ✅ 全 2 タスク成功
succeeded: ['task_1', 'task_2']
✅ テスト4 PASS

🎉 Phase 6 Skill 単体テスト 全 PASS


## 📈 Phase 5 → Phase 6 変化サマリー


In [23]:
print("""
╔══════════════════════╦══════════════════════╦══════════════════════════════════════╗
║ 項目                 ║ Phase 5              ║ Phase 6                              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 対応操作数           ║ 1 操作のみ           ║ ✅ 複数操作（混在）対応              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ タスク管理           ║ なし                 ║ ✅ DAG（有向非巡回グラフ）           ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 依存関係解決         ║ なし                 ║ ✅ トポロジカルソート                ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 制御レイヤー         ║ Worker のみ          ║ ✅ Orchestrator + Worker             ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 失敗時挙動           ║ Worker が rollback   ║ ✅ Orchestrator が残タスクを中断     ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ Reviewer             ║ Junos仕様ルールあり  ║ ✅ 意図整合性チェックのみ           ║
║                      ║ → 誤 REJECT リスク   ║   Junos仕様は validate_xml に移譲   ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ Skill数              ║ 7                    ║ ✅ 10（+3新規）                      ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ Worker 変更          ║ -                    ║ ✅ 変更なし（Phase 5 継承）          ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 並列化対応           ║ -                    ║ ✅ parallel フラグ（将来の基盤）     ║
╚══════════════════════╩══════════════════════╩══════════════════════════════════════╝
""")

print("\n🔮 Phase 7 以降への道筋:")
print("   - Worker の並列実行（parallel=True タスクを asyncio.gather で並列化）")
print("   - normalize_vlan_name_skill: inventory 結果で VLAN 名を正規化")
print("   - memory_skill: 過去の成功/失敗パターンを Embedding で記憶")



╔══════════════════════╦══════════════════════╦══════════════════════════════════════╗
║ 項目                 ║ Phase 5              ║ Phase 6                              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 対応操作数           ║ 1 操作のみ           ║ ✅ 複数操作（混在）対応              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ タスク管理           ║ なし                 ║ ✅ DAG（有向非巡回グラフ）           ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 依存関係解決         ║ なし                 ║ ✅ トポロジカルソート                ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 制御レイヤー         ║ Worker のみ          ║ ✅ Orchestrator + Worker             ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 失敗時挙動           ║ Worker が rollback   ║ ✅ Orchestrator が残タスクを中断     ║
╠══════════════════════╬═════════════

## 🛡️ Phase 7: 本番化補強レイヤ

Phase 6 の Orchestrator-Worker パターンに **4 つの安全境界レイヤ** を追加します。

```
補強マップ:
  ④ PolicyChecker      — AIが触ってよい範囲をYAMLで宣言的に制限
  ① ValidationAgent   — YANG構文 + 禁止コマンドの安全チェック
  ③ RollbackOrchestrator — candidate config 経由のトランザクション制御
  ⑤ AuditLogger       — Decision Trace をJSONLで永続保存
```

いずれも **既存の Worker / Skill を変更せず**、Orchestrator の `run()` に
フック・ポイントとして差し込む設計です。

## 🛡️ Phase 7 Step 1: PolicyChecker ④

Worker へのディスパッチ直前に、タスクと生成XMLをポリシーファイルと照合して **ブロック** します。

- ポリシーは `policy.yaml` で宣言的に管理（コード変更なしで運用調整可能）
- 違反があれば Orchestrator がそのタスクをスキップし、Decision Trace に記録

In [24]:
# ============================================================
# Phase 7 Step 1: PolicyChecker
# Orchestrator の Worker ディスパッチ直前に差し込む安全ゲート
# ============================================================

import yaml
import os
from dataclasses import dataclass, field as dc_field
from typing import List as TList

# ── デフォルトポリシー（policy.yaml が無い場合のフォールバック）──
DEFAULT_POLICY = {
    "allowed_interfaces": ["ge-0/0/0", "xe-0/0/1", "xe-0/0/2"],
    "allowed_vlan_ids": list(range(10, 201)),   # VLAN 10〜200
    "forbidden_keywords": [
        "delete-config", "kill-session", "restart",
        "delete system", "format"
    ],
    "allowed_netconf_nodes": ["vlans", "interfaces"],
    "max_vlan_operations_per_run": 20
}

POLICY_YAML_PATH = "./policy.yaml"


def _load_policy(path: str = POLICY_YAML_PATH) -> dict:
    """policy.yaml を読み込む。存在しなければ DEFAULT_POLICY を使用。"""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            loaded = yaml.safe_load(f)
            print(f"✅ [PolicyChecker] ポリシーファイル読み込み: {path}")
            return loaded
    print(f"⚠️  [PolicyChecker] {path} が見つかりません → デフォルトポリシーを使用")
    return DEFAULT_POLICY


@dataclass
class PolicyViolation:
    rule: str
    detail: str


def check_policy(
    task: dict,
    xml_config: str,
    policy: dict = None
) -> dict:
    """
    タスクと生成XMLをポリシーと照合する。

    Args:
        task:       task_decomposer が生成したタスク dict
        xml_config: Worker が生成した XML 文字列
        policy:     ポリシー dict（None なら自動ロード）

    Returns:
        {
            "allowed":    bool,
            "violations": [{"rule": str, "detail": str}, ...]
        }
    """
    if policy is None:
        policy = _load_policy()

    violations: TList[PolicyViolation] = []

    # ① インターフェース制限
    iface = task.get("interface")
    allowed_ifaces = policy.get("allowed_interfaces", [])
    if iface and allowed_ifaces and iface not in allowed_ifaces:
        violations.append(PolicyViolation(
            rule="interface_allowlist",
            detail=f"interface '{iface}' is not in allowed_interfaces: {allowed_ifaces}"
        ))

    # ② VLAN ID 制限
    vlan_id = task.get("vlan_id")
    allowed_ids = policy.get("allowed_vlan_ids", [])
    if vlan_id and allowed_ids:
        try:
            if int(vlan_id) not in allowed_ids:
                violations.append(PolicyViolation(
                    rule="vlan_id_allowlist",
                    detail=f"VLAN ID {vlan_id} is not in allowed_vlan_ids"
                ))
        except ValueError:
            pass  # vlan_id が数値でない場合はスキップ

    # ③ 禁止キーワードスキャン（XML 全体）
    if xml_config:
        for kw in policy.get("forbidden_keywords", []):
            if kw.lower() in xml_config.lower():
                violations.append(PolicyViolation(
                    rule="forbidden_keyword",
                    detail=f"forbidden keyword detected in XML: '{kw}'"
                ))

    # ④ 許可 NETCONF ノードチェック（ルート直下の要素）
    allowed_nodes = policy.get("allowed_netconf_nodes", [])
    if xml_config and allowed_nodes:
        try:
            import xml.etree.ElementTree as _ET
            root = _ET.fromstring(xml_config)
            for child in root:
                if child.tag not in allowed_nodes:
                    violations.append(PolicyViolation(
                        rule="netconf_node_allowlist",
                        detail=f"<{child.tag}> is not in allowed_netconf_nodes: {allowed_nodes}"
                    ))
        except Exception:
            pass  # XML パース失敗は validate_xml_structure が別途処理

    return {
        "allowed": len(violations) == 0,
        "violations": [{"rule": v.rule, "detail": v.detail} for v in violations]
    }


# ── policy.yaml のサンプルを自動生成（初回のみ）──
def write_default_policy_yaml(path: str = POLICY_YAML_PATH):
    """policy.yaml が存在しない場合にデフォルト内容で生成する。"""
    if os.path.exists(path):
        # 既存ファイルに !python タグが含まれていたら削除して再生成
        with open(path, encoding="utf-8") as _f:
            _content = _f.read()
        if "!python" in _content:
            os.remove(path)
            print(f"⚠️  {path} に非互換タグを検出 → 削除して再生成します")
        else:
            print(f"ℹ️  {path} は既に存在します（上書きしません）")
            return
    content = """# NETCONF Agent 操作ポリシー
# このファイルを編集することで、AIが操作できる範囲を制限できます。

# 操作を許可するインターフェース一覧
allowed_interfaces:
  - ge-0/0/0
  - xe-0/0/1
  - xe-0/0/2

# 操作を許可する VLAN ID の範囲（10〜200）
# yaml.safe_load 対応: !python タグ不使用、明示的なリストで記載
allowed_vlan_ids: [10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,
  26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,
  48,49,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200]

# XML 内に含まれると即ブロックするキーワード
forbidden_keywords:
  - delete-config
  - kill-session
  - restart
  - delete system
  - format

# 許可する NETCONF 設定ノード（<configuration> 直下の要素）
allowed_netconf_nodes:
  - vlans
  - interfaces

# 1回の実行で許可する最大 VLAN 操作数
max_vlan_operations_per_run: 20
"""
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"✅ {path} を生成しました")


write_default_policy_yaml()
GLOBAL_POLICY = _load_policy()
print("✅ PolicyChecker 定義完了")
print(f"   forbidden_keywords : {GLOBAL_POLICY.get('forbidden_keywords')}")
print(f"   allowed_netconf_nodes: {GLOBAL_POLICY.get('allowed_netconf_nodes')}")
print(f"   allowed_interfaces : {GLOBAL_POLICY.get('allowed_interfaces')}")


⚠️  ./policy.yaml に非互換タグを検出 → 削除して再生成します
✅ ./policy.yaml を生成しました
✅ [PolicyChecker] ポリシーファイル読み込み: ./policy.yaml
✅ PolicyChecker 定義完了
   forbidden_keywords : ['delete-config', 'kill-session', 'restart', 'delete system', 'format']
   allowed_netconf_nodes: ['vlans', 'interfaces']
   allowed_interfaces : ['ge-0/0/0', 'xe-0/0/1', 'xe-0/0/2']


## 🛡️ Phase 7 Step 2: ValidationAgent ①

既存の `validate_xml_structure`（Junosルール）に加え、**ネットワーク安全性観点**のチェックを追加します。

| チェック | 内容 |
|---|---|
| 危険パターン | `delete-config`, `copy-config → running` 等 |
| ループバック保護 | `lo0` / `loopback` への操作を禁止 |
| 大量削除検出 | 1リクエストで多数の VLAN 削除を警告 |
| 空XML | デプロイ前に確実にブロック |

In [25]:
# ============================================================
# Phase 7 Step 2: ValidationAgent（安全性チェック強化）
# 既存の validate_xml_structure の後段に追加するセーフティレイヤ
# ============================================================

import re as _re

# 危険なパターン定義
# (pattern, severity, message)
DANGEROUS_PATTERNS = [
    (r"<delete-config",         "error",   "<delete-config> is absolutely forbidden"),
    (r"<copy-config[^>]*running","error",   "copy-config targeting running is forbidden"),
    (r"kill-session",            "error",   "kill-session is forbidden"),
    (r"<interface[^>]*>\s*<name[^>]*>\s*lo0", "error", "loopback lo0 modification is forbidden"),
    (r"loopback",                "warning", "loopback interface reference detected — verify intent"),
    (r"<format",                 "error",   "format operation is forbidden"),
]

BULK_DELETE_THRESHOLD = 5  # 1リクエストでこれ以上の delete は警告


def validate_safety(
    xml_config: str,
    task: dict = None
) -> dict:
    """
    XML設定のネットワーク安全性チェック（Phase 7 新規）

    validate_xml_structure（構文チェック）の後段で呼ぶ。
    戻り値の 'safe' が False のとき、Orchestrator はデプロイをブロックする。

    Args:
        xml_config: 生成された XML 文字列
        task:       タスク dict（コンテキスト補完用、省略可）

    Returns:
        {
            "safe":     bool,
            "errors":   [str, ...],   # safe=False の原因
            "warnings": [str, ...]    # 続行可だが注意が必要
        }
    """
    if not xml_config or not xml_config.strip():
        return {"safe": False, "errors": ["XML is empty"], "warnings": []}

    errors = []
    warnings = []

    # ① 危険パターンスキャン
    for pattern, severity, message in DANGEROUS_PATTERNS:
        if _re.search(pattern, xml_config, _re.IGNORECASE | _re.DOTALL):
            if severity == "error":
                errors.append(message)
            else:
                warnings.append(message)

    # ② 大量削除検出
    delete_count = len(_re.findall(r'operation=["\']delete["\']', xml_config, _re.IGNORECASE))
    if delete_count >= BULK_DELETE_THRESHOLD:
        warnings.append(
            f"Bulk delete detected: {delete_count} delete operations in one request "
            f"(threshold={BULK_DELETE_THRESHOLD})"
        )

    # ③ タスクと XML の操作一貫性チェック（task が渡された場合）
    if task:
        op = task.get("operation", "")
        vlan_name = task.get("target_vlan", "")
        if op == "delete" and vlan_name:
            if f'operation="delete"' not in xml_config and "operation='delete'" not in xml_config:
                warnings.append(
                    f"Task says delete '{vlan_name}' but no delete operation found in XML"
                )
        elif op == "create" and vlan_name:
            if vlan_name not in xml_config:
                warnings.append(
                    f"Task says create '{vlan_name}' but VLAN name not found in XML"
                )

    return {
        "safe": len(errors) == 0,
        "errors": errors,
        "warnings": warnings
    }


print("✅ ValidationAgent 定義完了")
print(f"   危険パターン数: {len(DANGEROUS_PATTERNS)}")
print(f"   大量削除しきい値: {BULK_DELETE_THRESHOLD} 操作")


✅ ValidationAgent 定義完了
   危険パターン数: 6
   大量削除しきい値: 5 操作


## 🛡️ Phase 7 Step 3: RollbackOrchestrator ③

既存の `deploy_netconf_config` は `running` に直接コミットしています。
Phase 7 では `candidate` datastore を経由するトランザクション制御に切り替えます。

```
フロー:
  edit-config → candidate
  ↓
  commit confirmed (timeout=30s)  ← 30秒以内に確認がなければ自動ロールバック
  ↓
  差分確認 OK → commit (確定)
  差分確認 NG / 例外 → discard-changes (即時ロールバック)
```

> **注意**: デバイスが `candidate` datastore / `commit confirmed` をサポートしていない場合は
> 自動的に従来の `deploy_netconf_config`（running 直接）にフォールバックします。

In [26]:
# ============================================================
# Phase 7 Step 3: RollbackOrchestrator
# candidate config 経由のトランザクション制御デプロイ
# ============================================================

def deploy_with_candidate(
    xml_config: str,
    device_ip: str,
    username: str,
    password: str,
    port: str = "830",
    confirm_timeout: int = 30,
    comment: str = "AI Agent Phase 7 - candidate deploy"
) -> dict:
    """
    candidate datastore を経由した安全なデプロイ（Phase 7 新規）

    commit confirmed を使用することで、確認コミットがなければ
    confirm_timeout 秒後に自動ロールバックされる。

    デバイスが candidate / commit confirmed 非対応の場合は
    従来の deploy_netconf_config にフォールバックする。

    Returns:
        {
            "status":   "success" | "rolled_back" | "failure" | "skipped" | "fallback",
            "diff":     str,
            "message":  str,
            "method":   "candidate" | "direct" | "skipped"
        }
    """
    if not JUNOS_AVAILABLE:
        return {
            "status": "skipped", "diff": "",
            "message": "Junos modules not available",
            "method": "skipped"
        }

    try:
        with Device(host=device_ip, user=username, password=password, port=int(port)) as dev:
            cu = Config(dev)

            # candidate datastore サポート確認
            capabilities = dev.facts.get("2RE", None)  # フォールバック判定用
            try:
                cu.lock()  # candidate ロック取得
            except Exception as lock_err:
                # ロック取得失敗 → fallback to direct deploy
                cu_unlock_safe(cu)
                print(f"⚠️  [RollbackOrchestrator] candidate lock 失敗 → fallback: {lock_err}")
                return _fallback_deploy(xml_config, device_ip, username, password, port, comment)

            try:
                # ① candidate に書き込み
                try:
                    cu.load(xml_config, format="xml", merge=True)
                except ConfigLoadError as load_err:
                    severity = getattr(load_err, 'severity', 'error')
                    err_msg  = getattr(load_err, 'message', str(load_err))
                    cu.rollback()
                    cu_unlock_safe(cu)
                    if severity == 'warning' or 'statement not found' in err_msg.lower():
                        return {"status": "no_changes", "diff": "",
                                "message": f"No changes (target not found): {err_msg}",
                                "method": "candidate"}
                    return {"status": "failure", "diff": "",
                            "message": f"ConfigLoadError: {err_msg}",
                            "method": "candidate"}

                # ② 差分確認
                diff = cu.diff()
                if not diff:
                    cu.rollback()
                    cu_unlock_safe(cu)
                    return {"status": "no_changes", "diff": "",
                            "message": "No configuration changes detected",
                            "method": "candidate"}

                # ③ commit confirmed（自動ロールバックタイマー付き）
                try:
                    cu.commit(confirmed=True, timeout=confirm_timeout, comment=comment)
                except Exception:
                    # commit confirmed 非対応 → 通常 commit にフォールバック
                    cu.commit(comment=comment)
                    cu_unlock_safe(cu)
                    return {"status": "success", "diff": diff,
                            "message": f"Deployed (commit confirmed unsupported, used plain commit)",
                            "method": "candidate"}

                # ④ 確認コミット（タイマーを解除して確定）
                cu.commit(comment=f"{comment} - confirmed")
                cu_unlock_safe(cu)
                return {"status": "success", "diff": diff,
                        "message": f"Deployed via candidate + commit confirmed to {device_ip}",
                        "method": "candidate"}

            except Exception as e:
                # ⑤ 例外時は即 discard
                try:
                    cu.rollback()
                except Exception:
                    pass
                cu_unlock_safe(cu)
                return {"status": "rolled_back", "diff": "",
                        "message": f"Exception during deploy → auto rolled back: {e}",
                        "method": "candidate"}

    except Exception as e:
        return {"status": "failure", "diff": "",
                "message": f"Connection failed: {e}",
                "method": "candidate"}


def cu_unlock_safe(cu):
    """candidate ロックを安全に解放する（例外を握りつぶす）。"""
    try:
        cu.unlock()
    except Exception:
        pass


def _fallback_deploy(xml_config, device_ip, username, password, port, comment):
    """candidate 非対応デバイス向けのフォールバック（従来の直接 commit）。"""
    result = deploy_netconf_config(
        xml_config=xml_config,
        device_ip=device_ip,
        username=username,
        password=password,
        port=port,
        comment=comment
    )
    result["method"] = "direct"
    result["message"] = "[fallback] " + result.get("message", "")
    return result


# Phase 7 用 deploy Skill（candidate版）を登録
deploy_candidate_skill = Skill(
    name="deploy_netconf_candidate",
    description=(
        "Deploy XML config via NETCONF candidate datastore with commit confirmed. "
        "Auto-rollback if confirmation fails within timeout. "
        "Falls back to direct commit if candidate is unsupported. "
        "Returns status(success/rolled_back/no_changes/failure/skipped), diff, method."
    ),
    function=deploy_with_candidate,
    parameters={
        "xml_config":      {"type": "string"},
        "device_ip":       {"type": "string"},
        "username":        {"type": "string"},
        "password":        {"type": "string"},
        "port":            {"type": "string"},
        "confirm_timeout": {"type": "integer", "description": "seconds before auto-rollback (default 30)"}
    }
)

print("✅ RollbackOrchestrator 定義完了")
print("   commit confirmed タイムアウト: 30秒（デフォルト）")
print("   candidate 非対応デバイス: 自動フォールバック")


✅ RollbackOrchestrator 定義完了
   commit confirmed タイムアウト: 30秒（デフォルト）
   candidate 非対応デバイス: 自動フォールバック


## 🛡️ Phase 7 Step 4: AuditLogger ⑤

Phase 6 の `OrchestratorAgent.logs`（printログ）を、**Decision Trace 付き JSONL** として永続保存します。

各エントリに「なぜその操作を選んだか（rationale）」を記録することで、
後から人間がレビューできる監査証跡を実現します。

```json
{"timestamp": "2025-01-01T12:00:00", "task_id": "task_1",
 "skill": "deploy_netconf_candidate", "policy_checked": true,
 "safety_checked": true, "deploy_status": "success",
 "diff": "...", "violations": [], "rationale": "Create VLAN100"}
```

In [27]:
# ============================================================
# Phase 7 Step 4: AuditLogger
# Decision Trace を JSONL 形式で永続保存する監査ログ機構
# ============================================================

import json
from datetime import datetime
from pathlib import Path

AUDIT_LOG_PATH = "./audit_log.jsonl"


class AuditLogger:
    """
    Orchestrator の各タスク実行を構造化ログとして記録する（Phase 7 新規）

    1 タスク = 1 エントリ。JSONL 形式で append 保存。
    Decision Trace（why）を含むため、後からの監査・デバッグが可能。
    """

    def __init__(self, log_path: str = AUDIT_LOG_PATH):
        self.log_path = log_path
        self._session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._entries = []  # セッション中のエントリをメモリにも保持
        print(f"✅ AuditLogger 初期化: {log_path} (session={self._session_id})")

    def record(
        self,
        task: dict,
        worker_result: dict,
        policy_result: dict = None,
        safety_result: dict = None,
        deploy_method: str = "unknown"
    ):
        """
        1タスク分の実行結果を記録する。

        Args:
            task:          task_decomposer が生成したタスク dict
            worker_result: WorkerAgent の実行結果
            policy_result: PolicyChecker の結果（省略可）
            safety_result: ValidationAgent の結果（省略可）
            deploy_method: "candidate" | "direct" | "skipped" | "unknown"
        """
        deploy_status = worker_result.get("deployment_status") or {}
        audit_status  = worker_result.get("audit_status")  or {}

        # rationale: タスクの自然言語説明（decision の why）
        rationale = task.get("description", "") or (
            f"{task.get('operation','?')} {task.get('target_vlan','?')}"
            + (f" on {task.get('interface','')}" if task.get('interface') else "")
        )

        entry = {
            "session_id":      self._session_id,
            "timestamp":       datetime.now().isoformat(),
            "task_id":         task.get("id", "unknown"),
            "operation":       task.get("operation", "unknown"),
            "target_vlan":     task.get("target_vlan", ""),
            "interface":       task.get("interface", ""),
            "rationale":       rationale,
            # ポリシー・安全性チェック結果
            "policy_allowed":  (policy_result or {}).get("allowed", None),
            "policy_violations": (policy_result or {}).get("violations", []),
            "safety_passed":   (safety_result or {}).get("safe", None),
            "safety_errors":   (safety_result or {}).get("errors", []),
            "safety_warnings": (safety_result or {}).get("warnings", []),
            # デプロイ結果
            "deploy_method":   deploy_method,
            "deploy_status":   deploy_status.get("status", "unknown"),
            "diff":            deploy_status.get("diff", ""),
            "deploy_message":  deploy_status.get("message", ""),
            # 監査・ロールバック結果
            "audit_status":    audit_status.get("status", ""),
            "audit_message":   audit_status.get("message", ""),
            "rollback_status": (worker_result.get("rollback_status") or {}).get("status", ""),
        }

        self._entries.append(entry)
        self._write(entry)

    def _write(self, entry: dict):
        """JSONL 形式で 1 エントリ追記する。"""
        try:
            with open(self.log_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        except Exception as e:
            print(f"⚠️  [AuditLogger] ログ書き込み失敗: {e}")

    def record_blocked(self, task: dict, reason: str, violations: list = None):
        """PolicyChecker / ValidationAgent にブロックされたタスクを記録する。"""
        entry = {
            "session_id":  self._session_id,
            "timestamp":   datetime.now().isoformat(),
            "task_id":     task.get("id", "unknown"),
            "operation":   task.get("operation", "unknown"),
            "target_vlan": task.get("target_vlan", ""),
            "interface":   task.get("interface", ""),
            "rationale":   task.get("description", ""),
            "deploy_status": "blocked",
            "deploy_message": reason,
            "policy_violations": violations or [],
        }
        self._entries.append(entry)
        self._write(entry)

    def summary(self) -> dict:
        """セッション中の集計サマリーを返す。"""
        total    = len(self._entries)
        success  = sum(1 for e in self._entries if e.get("deploy_status") == "success")
        blocked  = sum(1 for e in self._entries if e.get("deploy_status") == "blocked")
        failed   = sum(1 for e in self._entries if e.get("deploy_status") in ("failure", "rolled_back"))
        return {
            "total": total, "success": success,
            "blocked": blocked, "failed": failed,
            "skipped": total - success - blocked - failed
        }

    def print_summary(self):
        s = self.summary()
        print("\n" + "=" * 50)
        print("📋 AuditLogger セッションサマリー")
        print("=" * 50)
        print(f"  総タスク数 : {s['total']}")
        print(f"  成功       : {s['success']}")
        print(f"  ブロック   : {s['blocked']}")
        print(f"  失敗/RB    : {s['failed']}")
        print(f"  スキップ   : {s['skipped']}")
        print(f"  ログファイル: {self.log_path}")


print("✅ AuditLogger 定義完了")
print(f"   ログ保存先: {AUDIT_LOG_PATH}")


✅ AuditLogger 定義完了
   ログ保存先: ./audit_log.jsonl


## 🛡️ Phase 7 Step 5: OrchestratorAgent Phase 7

Phase 6 の `OrchestratorAgent` を継承し、4つの補強レイヤを **Worker ディスパッチ直前・直後** に組み込みます。

```
Worker ディスパッチフロー（Phase 7）:
  [Step 0] get_inventory         ← 変更なし
  [Step 1] task_decomposer       ← 変更なし
  [Step 2] dependency_resolver   ← 変更なし
  [Step 3] Worker ディスパッチ   ← ここに補強を追加
           ├─ ① validate_safety  (ValidationAgent)
           ├─ ④ check_policy     (PolicyChecker)    ← ブロックゲート
           ├─ Worker.run()       (既存)
           │   └─ deploy_with_candidate (RollbackOrchestrator)
           └─ ⑤ audit_logger.record()  (AuditLogger)
  [Step 4] result_aggregator     ← 変更なし
  [Step 5] audit_logger.print_summary() ← 新規
```

In [28]:
# ============================================================
# Phase 7: OrchestratorAgentV7
# Phase 6 OrchestratorAgent を継承し 4 補強レイヤを統合
# ============================================================

class OrchestratorAgentV7(OrchestratorAgent):
    """
    本番化補強版 Orchestrator（Phase 7 新規）

    Phase 6 OrchestratorAgent を継承し、Worker ディスパッチ前後に
    以下の補強レイヤを追加する:
      ① ValidationAgent  — XML 安全性チェック
      ④ PolicyChecker    — 操作ポリシーによるブロック
      ③ RollbackOrch.    — candidate config 経由デプロイ（Worker内で使用）
      ⑤ AuditLogger      — Decision Trace の JSONL 永続保存

    Worker (NetconfRagWorkflowPhase3) 自体は変更しない。
    ただし Worker が deploy_netconf を呼ぶ前に、
    Orchestrator 側で policy/safety チェックを行い、
    通過したタスクのみ Worker に渡す。
    """

    def __init__(
        self,
        retriever,
        llm,
        skills=None,
        max_retries: int = 3,
        max_review_rounds: int = 2,
        policy: dict = None,
        audit_log_path: str = AUDIT_LOG_PATH,
        use_candidate_deploy: bool = True,
    ):
        super().__init__(
            retriever=retriever, llm=llm, skills=skills,
            max_retries=max_retries, max_review_rounds=max_review_rounds
        )
        self.policy = policy or _load_policy()
        self.audit_logger = AuditLogger(log_path=audit_log_path)
        self.use_candidate_deploy = use_candidate_deploy
        print(f"   use_candidate_deploy : {use_candidate_deploy}")
        print(f"   audit_log_path       : {audit_log_path}")

    # ------------------------------------------------------------------
    # _dispatch_task: Worker へのディスパッチを補強したオーバーライドメソッド
    # ------------------------------------------------------------------
    async def _dispatch_task(
        self,
        task: dict,
        idx: int,
        total: int,
        device_ip: str,
        username: str,
        password: str,
        port: str,
        deploy: bool,
        generated_xml: str = None
    ) -> dict:
        """
        Phase 7 拡張ディスパッチ:
          1. XML が生成済みなら ① validate_safety → ④ check_policy
          2. いずれかが NG なら Worker を呼ばずブロック
          3. OK なら Worker を実行
          4. ⑤ AuditLogger に結果を記録
        """
        tid = task.get("id", "?")
        policy_result  = None
        safety_result  = None
        deploy_method  = "unknown"

        # ── skip タスクは即完了 ──
        if task.get("operation") == "skip":
            self.log(f"  ⏭️  [{idx}/{total}] SKIP: {tid}")
            skipped_result = {
                "validation_status": True,
                "deployment_status": {"status": "no_changes", "diff": "",
                                       "message": task.get('description', 'skipped')},
                "audit_status": None, "rollback_status": None
            }
            self.audit_logger.record(task, skipped_result,
                                     deploy_method="skipped")
            return skipped_result

        # ── Worker でまず XML を生成（deploy=False で生成のみ）──
        worker_query = self._build_worker_query(task)
        worker = NetconfRagWorkflowPhase3(
            retriever=self.retriever, llm=self.llm,
            skills=ALL_SKILLS,
            max_retries=self.max_retries,
            max_review_rounds=self.max_review_rounds,
        )
        worker.xml_reviewer.instructions = REVIEWER_INSTRUCTIONS_V6

        # XML 生成フェーズ（deploy=False → NETCONFには送らない）
        try:
            gen_result = await worker.run(
                user_query=worker_query,
                device_ip=device_ip, username=username,
                password=password, port=port,
                deploy=False  # ← 生成のみ
            )
        except Exception as e:
            self.log(f"  ❌ XML生成エラー: {e}")
            err_result = {
                "validation_status": False,
                "deployment_status": {"status": "failure", "diff": "", "message": str(e)},
                "audit_status": None, "rollback_status": None
            }
            self.audit_logger.record_blocked(task, f"XML generation failed: {e}")
            return err_result

        xml_config = gen_result.get("final_xml", "")

        # ── ① ValidationAgent ──
        self.log(f"  🔍 [ValidationAgent] 安全性チェック: {tid}")
        safety_result = validate_safety(xml_config, task=task)
        if not safety_result["safe"]:
            msg = f"ValidationAgent BLOCK: {safety_result['errors']}"
            self.log(f"  🚫 {msg}")
            self.audit_logger.record_blocked(task, msg,
                                              violations=safety_result["errors"])
            return {
                "validation_status": False,
                "deployment_status": {"status": "blocked", "diff": "", "message": msg},
                "audit_status": None, "rollback_status": None
            }
        if safety_result["warnings"]:
            self.log(f"  ⚠️  [ValidationAgent] warnings: {safety_result['warnings']}")

        # ── ④ PolicyChecker ──
        self.log(f"  🛡️  [PolicyChecker] ポリシーチェック: {tid}")
        policy_result = check_policy(task, xml_config, policy=self.policy)
        if not policy_result["allowed"]:
            msg = f"PolicyChecker BLOCK: {policy_result['violations']}"
            self.log(f"  🚫 {msg}")
            self.audit_logger.record_blocked(task, msg,
                                              violations=policy_result["violations"])
            return {
                "validation_status": False,
                "deployment_status": {"status": "blocked", "diff": "", "message": msg},
                "audit_status": None, "rollback_status": None
            }
        self.log(f"  ✅ Policy OK / Safety OK → Worker デプロイへ")

        # ── Worker 本番デプロイ（candidate deploy 切り替え）──
        if deploy and self.use_candidate_deploy:
            # candidate 経由で直接デプロイ
            self.log(f"  🔄 [RollbackOrchestrator] candidate deploy: {tid}")
            deploy_res = deploy_with_candidate(
                xml_config=xml_config,
                device_ip=device_ip, username=username,
                password=password, port=port
            )
            deploy_method = deploy_res.get("method", "candidate")
            # audit は既存 Skill を流用
            if deploy_res.get("status") == "success":
                audit_res = audit_deployment(
                    xml_config=xml_config,
                    device_ip=device_ip, username=username,
                    password=password, port=port
                )
            else:
                audit_res = None

            worker_result = {
                "final_xml": xml_config,
                "validation_status": safety_result["safe"],
                "deployment_status": deploy_res,
                "audit_status": audit_res,
                "rollback_status": None
            }
        else:
            # deploy=False または use_candidate_deploy=False → Worker の通常フロー
            try:
                worker_result = await worker.run(
                    user_query=worker_query,
                    device_ip=device_ip, username=username,
                    password=password, port=port,
                    deploy=deploy
                )
                deploy_method = "direct"
            except Exception as e:
                self.log(f"  ❌ Worker エラー: {e}")
                worker_result = {
                    "validation_status": False,
                    "deployment_status": {"status": "failure", "diff": "", "message": str(e)},
                    "audit_status": None, "rollback_status": None
                }

        # ── ⑤ AuditLogger ──
        self.audit_logger.record(
            task=task,
            worker_result=worker_result,
            policy_result=policy_result,
            safety_result=safety_result,
            deploy_method=deploy_method
        )

        return worker_result

    # ------------------------------------------------------------------
    # run: Phase 6 の run() を override して _dispatch_task を呼ぶ
    # ------------------------------------------------------------------
    async def run(
        self,
        user_query: str,
        device_ip: str = None,
        username: str = None,
        password: str = None,
        port: str = "830",
        deploy: bool = False,
    ) -> dict:
        """
        Phase 7 メインフロー（Phase 6 run() を拡張）

        Step 0〜2 は Phase 6 と同じ（get_inventory / task_decomposer /
        dependency_resolver）。
        Step 3 のみ _dispatch_task() を呼ぶ形に変更。
        """
        self.logs = []
        self.log("\n" + "=" * 70)
        self.log("🛡️  Phase 7: OrchestratorAgentV7 起動")
        self.log("=" * 70)
        self.log(f"入力クエリ: {user_query}")

        result = {
            "user_query": user_query,
            "tasks": [], "execution_order": [],
            "task_results": [], "aggregated": {},
            "orchestrator_logs": self.logs
        }

        # ── Step 0: get_inventory ──
        orchestrator_inventory = None
        if all([device_ip, username, password]):
            self.log("\n" + "─" * 50)
            self.log("🗂️  [Step 0] get_inventory")
            inv = self._run_skill(
                "get_inventory",
                device_ip=device_ip, username=username,
                password=password, port=port
            )
            if inv and inv.get("status") == "success":
                orchestrator_inventory = inv
                self.log(f"   現在のVLAN一覧: {inv.get('vlan_names', [])}")
                result["orchestrator_inventory"] = inv
            else:
                self.log(f"   ⚠️  Inventory取得失敗（続行）")

        # ── Step 1: task_decomposer ──
        self.log("\n" + "─" * 50)
        self.log("📋 [Step 1] task_decomposer")
        decompose_result = self._run_skill(
            "task_decomposer", user_query=user_query,
            llm=self.llm, inventory=orchestrator_inventory
        )
        if not decompose_result or decompose_result["status"] != "success":
            msg = (decompose_result or {}).get("message", "Skill error")
            self.log(f"❌ task_decomposer 失敗: {msg}")
            result["aggregated"] = {"status": "all_failure",
                                    "summary": f"task_decomposer 失敗: {msg}",
                                    "report_lines": []}
            return result

        tasks = decompose_result["tasks"]
        result["tasks"] = tasks
        self.log(f"   → {len(tasks)} タスク検出")

        # ── Step 2: dependency_resolver ──
        self.log("\n" + "─" * 50)
        self.log("🔗 [Step 2] dependency_resolver")
        resolve_result = self._run_skill("dependency_resolver", tasks=tasks)
        if not resolve_result or resolve_result["status"] != "success":
            msg = (resolve_result or {}).get("message", "Skill error")
            self.log(f"❌ dependency_resolver 失敗: {msg}")
            result["aggregated"] = {"status": "all_failure",
                                    "summary": f"dependency_resolver 失敗: {msg}",
                                    "report_lines": []}
            return result

        execution_order = resolve_result["execution_order"]
        result["execution_order"] = execution_order
        self.log(f"   → 実行順: {[t['id'] for t in execution_order]}")

        # ── Step 3: 補強版 Worker ディスパッチ ──
        self.log("\n" + "─" * 50)
        self.log(f"🚀 [Step 3] Worker ディスパッチ（Phase 7 補強版）: {len(execution_order)} タスク")
        task_results = []

        for idx, task in enumerate(execution_order, 1):
            tid = task["id"]
            self.log(f"\n  [{idx}/{len(execution_order)}] 🔧 {tid}: "
                     f"{task.get('operation','?')} / {task.get('target_vlan','?')}")

            worker_result = await self._dispatch_task(
                task=task, idx=idx, total=len(execution_order),
                device_ip=device_ip, username=username,
                password=password, port=port, deploy=deploy
            )

            deploy_status = worker_result.get("deployment_status") or {}
            audit_status  = (worker_result.get("audit_status")  or {}).get("status", "")
            deploy_ok = deploy_status.get("status") in ("success", "no_changes", "skipped", "blocked")
            audit_ok  = audit_status in ("success", "skipped", "")
            task_success = deploy_ok and audit_ok

            ds = deploy_status.get('status', '?')
            self.log(f"  {'✅' if task_success else '❌'} {tid} → {ds}")
            task_results.append({"task": task, "result": worker_result})

            # 失敗（blocked 以外）時は残タスクを中断
            if not task_success and ds not in ("blocked", "no_changes"):
                self.log(f"\n  🛑 タスク {tid} 失敗 → 残タスクを中断")
                for rem in execution_order[idx:]:
                    task_results.append({
                        "task": rem,
                        "result": {
                            "validation_status": False,
                            "deployment_status": {
                                "status": "skipped",
                                "diff": "",
                                "message": f"Skipped due to failure of {tid}"
                            },
                            "audit_status": None, "rollback_status": None
                        }
                    })
                break

        result["task_results"] = task_results

        # ── Step 4: result_aggregator ──
        self.log("\n" + "─" * 50)
        self.log("📊 [Step 4] result_aggregator")
        aggregated = self._run_skill("result_aggregator", task_results=task_results)
        result["aggregated"] = aggregated or {}
        if aggregated:
            self.log(f"\n{aggregated['summary']}")

        # ── Step 5: AuditLogger サマリー ──
        self.audit_logger.print_summary()

        self.log("\n" + "=" * 70)
        self.log("🛡️  OrchestratorAgentV7 完了")
        self.log("=" * 70)

        return result


print("✅ OrchestratorAgentV7 クラス定義完了")
print("   継承元  : OrchestratorAgent (Phase 6)")
print("   追加機能: ValidationAgent / PolicyChecker / RollbackOrchestrator / AuditLogger")


✅ OrchestratorAgentV7 クラス定義完了
   継承元  : OrchestratorAgent (Phase 6)
   追加機能: ValidationAgent / PolicyChecker / RollbackOrchestrator / AuditLogger


## ⚙️ Phase 7 設定

In [29]:
# Phase 7 設定（Phase 6 設定を継承）
# DEVICE_CONFIG / DEFAULT_MODEL / MAX_RETRIES は Phase 6 セルで定義済み

# Phase 7 で追加する設定値
AUDIT_LOG_PATH_V7   = "./audit_log.jsonl"
USE_CANDIDATE_DEPLOY = True   # True: candidate経由 / False: 直接commit（デバイス対応状況に合わせて変更）

print("✅ Phase 7 設定")
print(f"   AUDIT_LOG_PATH     : {AUDIT_LOG_PATH_V7}")
print(f"   USE_CANDIDATE_DEPLOY: {USE_CANDIDATE_DEPLOY}")


✅ Phase 7 設定
   AUDIT_LOG_PATH     : ./audit_log.jsonl
   USE_CANDIDATE_DEPLOY: True


## 🚀 Phase 7 OrchestratorAgentV7 実行

In [30]:
# Phase 7 Orchestrator 実行
DEPLOY_FLAG = True  # または False
orchestrator_v7 = OrchestratorAgentV7(
    retriever=retriever,
    llm=llm,
    skills=ALL_SKILLS_V6,
    max_retries=MAX_RETRIES,
    max_review_rounds=MAX_REVIEW_ROUNDS,
    audit_log_path=AUDIT_LOG_PATH_V7,
    use_candidate_deploy=USE_CANDIDATE_DEPLOY,
)

orch_result_v7 = await orchestrator_v7.run(
        user_query=USER_QUERY_P6,  # ← USER_QUERY_P6 に修正
        device_ip=DEVICE_CONFIG['ip'],
        username=DEVICE_CONFIG['username'],
        password=DEVICE_CONFIG['password'],
        port=DEVICE_CONFIG['port'],
        deploy=DEPLOY_TO_DEVICE_P6,
    
)


✅ OrchestratorAgent 初期化完了
   登録 Skills: ['validate_xml', 'fix_xml', 'deploy_netconf', 'rollback', 'audit', 'get_inventory', 'lookup_documentation', 'task_decomposer', 'dependency_resolver', 'result_aggregator']
✅ [PolicyChecker] ポリシーファイル読み込み: ./policy.yaml
✅ AuditLogger 初期化: ./audit_log.jsonl (session=20260320_033502)
   use_candidate_deploy : True
   audit_log_path       : ./audit_log.jsonl

🛡️  Phase 7: OrchestratorAgentV7 起動
入力クエリ: SALES_VLANのVLAN ID 70を削除して、DEV_VLANのVLAN ID 200を追加してください。

──────────────────────────────────────────────────
🗂️  [Step 0] get_inventory
  🛠️  [Skill:get_inventory] 実行中...
  ✅ [Skill:get_inventory] 完了
   現在のVLAN一覧: ['DEV_VLAN', 'MGMT_VLAN']

──────────────────────────────────────────────────
📋 [Step 1] task_decomposer
  🛠️  [Skill:task_decomposer] 実行中...
  ✅ [Skill:task_decomposer] 完了
   → 2 タスク検出

──────────────────────────────────────────────────
🔗 [Step 2] dependency_resolver
  🛠️  [Skill:dependency_resolver] 実行中...
  ✅ [Skill:dependency_resolver] 完了
 

## 📊 Phase 7 実行結果

In [31]:
print("\n" + "=" * 70)
print("📊 Phase 7 OrchestratorAgentV7 実行サマリー")
print("=" * 70)

agg = orch_result_v7.get("aggregated", {})
print(f"\n{agg.get('summary', '(集計なし)')}")
for line in agg.get("report_lines", []):
    print(line)

# AuditLogger サマリー（再表示）
orchestrator_v7.audit_logger.print_summary()

# ブロックされたタスクの確認
blocked = [
    r for r in orch_result_v7.get("task_results", [])
    if (r["result"].get("deployment_status") or {}).get("status") == "blocked"
]
if blocked:
    print("\n🚫 ブロックされたタスク:")
    for b in blocked:
        t = b["task"]
        msg = (b["result"].get("deployment_status") or {}).get("message", "")
        print(f"   {t.get('id','?')}: {t.get('operation','?')} {t.get('target_vlan','?')} → {msg}")

print(f"\n📋 監査ログ: {AUDIT_LOG_PATH_V7}")



📊 Phase 7 OrchestratorAgentV7 実行サマリー

✅ 全 2 タスク成功
📊 Phase 6 Orchestrator 実行レポート

  📋 [task_1] delete / SALES_VLAN
     説明: Delete SALES_VLAN
     検証: ✅
     デプロイ: no_changes
     → ✅ SUCCESS

  📋 [task_2] skip / DEV_VLAN
     説明: Already exists — skip
     検証: ✅
     デプロイ: no_changes
     → ✅ SUCCESS

  ✅ 全 2 タスク成功

📋 AuditLogger セッションサマリー
  総タスク数 : 2
  成功       : 0
  ブロック   : 0
  失敗/RB    : 0
  スキップ   : 2
  ログファイル: ./audit_log.jsonl

📋 監査ログ: ./audit_log.jsonl


## 🔬 Phase 7 単体テスト

PolicyChecker・ValidationAgent を独立テストします。

In [32]:
print("\n" + "=" * 70)
print("🔬 Phase 7 単体テスト")
print("=" * 70)

# ── テスト 1: PolicyChecker — 正常（許可されたVLAN）──
print("\n--- テスト1: PolicyChecker 正常ケース ---")
result_p1 = check_policy(
    task={"id": "task_1", "operation": "create", "target_vlan": "VLAN100",
          "vlan_id": "100", "interface": None},
    xml_config="<configuration><vlans><vlan><name>VLAN100</name><vlan-id>100</vlan-id></vlan></vlans></configuration>",
    policy=GLOBAL_POLICY
)
print(f"  allowed: {result_p1['allowed']} (期待値: True)")
print(f"  violations: {result_p1['violations']}")

# ── テスト 2: PolicyChecker — ブロック（禁止キーワード）──
print("\n--- テスト2: PolicyChecker ブロックケース（禁止キーワード）---")
result_p2 = check_policy(
    task={"id": "task_2", "operation": "delete", "target_vlan": "VLAN10",
          "vlan_id": None, "interface": None},
    xml_config="<configuration><delete-config/></configuration>",
    policy=GLOBAL_POLICY
)
print(f"  allowed: {result_p2['allowed']} (期待値: False)")
print(f"  violations: {result_p2['violations']}")

# ── テスト 3: PolicyChecker — ブロック（許可外VLAN ID）──
print("\n--- テスト3: PolicyChecker ブロックケース（許可外VLAN ID）---")
result_p3 = check_policy(
    task={"id": "task_3", "operation": "create", "target_vlan": "VLAN999",
          "vlan_id": "999", "interface": None},
    xml_config="<configuration><vlans><vlan><name>VLAN999</name><vlan-id>999</vlan-id></vlan></vlans></configuration>",
    policy=GLOBAL_POLICY
)
print(f"  allowed: {result_p3['allowed']} (期待値: False)")
print(f"  violations: {result_p3['violations']}")

# ── テスト 4: ValidationAgent — 正常 ──
print("\n--- テスト4: ValidationAgent 正常ケース ---")
result_v1 = validate_safety(
    "<configuration><vlans><vlan operation='delete'><name>VLAN70</name></vlan></vlans></configuration>",
    task={"operation": "delete", "target_vlan": "VLAN70"}
)
print(f"  safe: {result_v1['safe']} (期待値: True)")
print(f"  errors:   {result_v1['errors']}")
print(f"  warnings: {result_v1['warnings']}")

# ── テスト 5: ValidationAgent — 危険パターン ──
print("\n--- テスト5: ValidationAgent 危険パターン ---")
result_v2 = validate_safety(
    "<rpc><delete-config><target><running/></target></delete-config></rpc>"
)
print(f"  safe: {result_v2['safe']} (期待値: False)")
print(f"  errors: {result_v2['errors']}")

# ── テスト 6: AuditLogger 単体動作確認 ──
print("\n--- テスト6: AuditLogger ---")
test_logger = AuditLogger(log_path="/tmp/test_audit.jsonl")
test_logger.record_blocked(
    task={"id": "test_task", "operation": "create", "target_vlan": "VLAN999",
          "description": "Create VLAN999 (blocked by policy)"},
    reason="VLAN ID 999 not in allowed_vlan_ids",
    violations=[{"rule": "vlan_id_allowlist", "detail": "VLAN ID 999 not allowed"}]
)
test_logger.print_summary()
import os
if os.path.exists("/tmp/test_audit.jsonl"):
    with open("/tmp/test_audit.jsonl") as f:
        print(f"  ログ内容: {f.read().strip()}")

print("\n✅ Phase 7 単体テスト完了")



🔬 Phase 7 単体テスト

--- テスト1: PolicyChecker 正常ケース ---
  allowed: True (期待値: True)
  violations: []

--- テスト2: PolicyChecker ブロックケース（禁止キーワード）---
  allowed: False (期待値: False)
  violations: [{'rule': 'forbidden_keyword', 'detail': "forbidden keyword detected in XML: 'delete-config'"}, {'rule': 'netconf_node_allowlist', 'detail': "<delete-config> is not in allowed_netconf_nodes: ['vlans', 'interfaces']"}]

--- テスト3: PolicyChecker ブロックケース（許可外VLAN ID）---
  allowed: False (期待値: False)
  violations: [{'rule': 'vlan_id_allowlist', 'detail': 'VLAN ID 999 is not in allowed_vlan_ids'}]

--- テスト4: ValidationAgent 正常ケース ---
  safe: True (期待値: True)
  errors:   []
  warnings: []

--- テスト5: ValidationAgent 危険パターン ---
  safe: False (期待値: False)
  errors: ['<delete-config> is absolutely forbidden']

--- テスト6: AuditLogger ---
✅ AuditLogger 初期化: /tmp/test_audit.jsonl (session=20260320_033553)

📋 AuditLogger セッションサマリー
  総タスク数 : 1
  成功       : 0
  ブロック   : 1
  失敗/RB    : 0
  スキップ   : 0
  ログファイル: /tmp/test_au

## 📈 Phase 6 → Phase 7 変化サマリー

In [33]:
print("""
╔══════════════════════╦══════════════════════╦══════════════════════════════════════╗
║ 項目                 ║ Phase 6              ║ Phase 7                              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 安全ゲート           ║ なし                 ║ PolicyChecker + ValidationAgent       ║
║ デプロイ方式         ║ running 直接 commit  ║ candidate + commit confirmed          ║
║ 自動ロールバック     ║ 例外時のみ           ║ タイムアウト + discard-changes         ║
║ 監査ログ             ║ print のみ           ║ JSONL (Decision Trace付き)            ║
║ ポリシー管理         ║ なし                 ║ policy.yaml (宣言的・コード変更不要) ║
║ Worker 変更          ║ —                    ║ 変更なし（継承・フック追加のみ）      ║
╚══════════════════════╩══════════════════════╩══════════════════════════════════════╝
""")



╔══════════════════════╦══════════════════════╦══════════════════════════════════════╗
║ 項目                 ║ Phase 6              ║ Phase 7                              ║
╠══════════════════════╬══════════════════════╬══════════════════════════════════════╣
║ 安全ゲート           ║ なし                 ║ PolicyChecker + ValidationAgent       ║
║ デプロイ方式         ║ running 直接 commit  ║ candidate + commit confirmed          ║
║ 自動ロールバック     ║ 例外時のみ           ║ タイムアウト + discard-changes         ║
║ 監査ログ             ║ print のみ           ║ JSONL (Decision Trace付き)            ║
║ ポリシー管理         ║ なし                 ║ policy.yaml (宣言的・コード変更不要) ║
║ Worker 変更          ║ —                    ║ 変更なし（継承・フック追加のみ）      ║
╚══════════════════════╩══════════════════════╩══════════════════════════════════════╝

